In [ ]:
# Hero figure. Public scores are the board's. Forecasts come from a model fitted on scored files (see the text), written down
# before each of our submissions; for other authors' files they are leave-one-out forecasts.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# our ladder (public LB) and the forecast made before each step, in points (1 point = 0.00001) from the file it starts from
PATH = [('version 6\n(step 9)', 0.95421, None), ('step 10\nstack in', 0.95426, 6.4), ('step 11\n(version 8)', 0.95427, 1.0),
        ('step 12\n(version 9)', 0.95430, 3.05), ('step 13\n(this version)', 0.95432, 2.57)]
TOP = {'Areeb Ali, Malyshev Danil': 0.95431, 'Asterios Panoras': 0.95428}      # top public notebooks on 17 Nov afternoon, before this version
# 16 files other authors scored on 16 Nov: (board change, leave-one-out forecast), points from our 0.95421 file
OTHERS = [(2, 2.01), (3, 2.85), (5, 4.04), (-1, -1.07), (-3, -2.18), (0, -0.27), (0, -0.04), (0, -0.22), (-2, -1.99), (-12, -12.03),
          (2, 1.43), (1, 0.75), (0, -0.35), (-2, -2.39), (-1, -0.24), (1, 1.53),
          (1, 0.6), (1, 0.9),                                                   # 17 Nov morning: the two 0.95428 files, from our 0.95427 file
          (1, 0.8), (-1, -0.9)]                                                 # 17 Nov afternoon: Areeb Ali's 0.95431 and 0.95429, from our 0.95430 file
# our four probes of this version: (board change, forecast), points from the file each starts from
OURS = {'step 10, balanced': (3, 5.0), 'step 10': (5, 6.4), 'step 11': (1, 1.0), 'step 11 + 20% BhuvanNR file': (2, 1.6), 'step 12': (3, 3.05),
        'step 13': (2, 2.57), 'step 13, larger': (2, 2.77), 'step 13 + CV step': (-1, -0.34)}
e = np.array([f - b for b, f in OTHERS]); RMSE16 = np.sqrt((e ** 2).mean())

SURFACE, PAGE = '#fcfcfb', '#f9f9f7'
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
GRID, AXIS, RING = '#e1e0d9', '#c3c2b7', '#e6e5df'
BLUE, ORANGE = '#2a78d6', '#eb6834'
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10, 'text.color': INK,
    'axes.edgecolor': AXIS, 'axes.labelcolor': INK2, 'axes.facecolor': SURFACE,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.8,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'axes.grid': False, 'figure.facecolor': PAGE, 'savefig.facecolor': PAGE,
})

def tile(fig, x, y0, w, h, label, value, note):
    ax = fig.add_axes([x, y0, w, h]); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle='round,pad=0,rounding_size=0.06', transform=ax.transAxes,
                                facecolor=SURFACE, edgecolor=RING, lw=1))
    ax.text(0.06, 0.74, label, transform=ax.transAxes, fontsize=10, color=INK2, va='center')
    ax.text(0.06, 0.42, value, transform=ax.transAxes, fontsize=21, fontweight='bold', color=INK, va='center')
    ax.text(0.06, 0.14, note, transform=ax.transAxes, fontsize=9, color=MUTED, va='center')

def head(ax, title, sub):
    ax.text(0, 1.10, title, transform=ax.transAxes, fontsize=12.5, fontweight='bold', color=INK, va='bottom')
    ax.text(0, 1.035, sub, transform=ax.transAxes, fontsize=9.5, color=INK2, va='bottom')

fig = plt.figure(figsize=(16, 9.6), dpi=110)
fig.text(0.035, 0.952, 'LM Ladder: ladder meets stack', fontsize=23, fontweight='bold', color=INK, va='center')
fig.text(0.035, 0.905, "Twenty files scored by other authors became new equations for our public model. With them, alexchilton's "
         "shallow-tree stack joins the ladder: +11 points in four steps.", fontsize=12, color=INK2, va='center')
tw, gap = 0.2175, 0.0125
TILES = [('Public LB of this version', '0.95432', 'version 9: 0.95430; version 6: 0.95421'),
         ('Steps 10 to 13 on the board', '+11 pts', 'forecast +13.0, written down first'),
         ('Scored files in the model', '105', '20 of them from six notebooks, 16-17 Nov'),
         ('Forecast error, 20 new files', f'{RMSE16:.1f} pts', 'each forecast with that file left out')]
for i, t in enumerate(TILES):
    tile(fig, 0.035 + i * (tw + gap), 0.75, tw, 0.11, *t)

# left: the ladder against the top public notebooks
ax = fig.add_axes([0.065, 0.08, 0.42, 0.55])
xs = np.arange(len(PATH)); pub = np.array([p for _, p, _ in PATH]) * 1e5 - 95400
for name, lb in TOP.items():
    yv = lb * 1e5 - 95400
    ax.axhline(yv, color=ORANGE, lw=1, ls=(0, (4, 3)), alpha=0.75, zorder=1)
    ax.text(-0.62, yv + 0.12, f'{name} {lb:.5f}', fontsize=8.8, color=ORANGE, va='bottom', ha='left')
for i, (_, _, fc) in enumerate(PATH):
    if fc is not None:
        f = pub[i - 1] + fc
        ax.plot([xs[i], xs[i]], [f, pub[i]], color=MUTED, lw=1.2, zorder=3)
        ax.plot([xs[i]], [f], marker='o', ms=11, mfc=SURFACE, mec=MUTED, mew=1.8, zorder=4)
ax.plot(xs, pub, color=BLUE, lw=2.5, marker='o', ms=9, mec=SURFACE, mew=1.5, zorder=5)
for i, (_, lb, _) in enumerate(PATH):
    ax.annotate(f'{lb:.5f}', (xs[i], pub[i]), xytext=(12, -4), textcoords='offset points', fontsize=9.5, color=INK, ha='left', va='top',
                fontweight='bold' if i == len(PATH) - 1 else 'normal')
ax.set_xticks(xs); ax.set_xticklabels([n for n, _, _ in PATH]); ax.tick_params(axis='x', labelsize=9.5, colors=INK2, length=0)
ax.set_xlim(-0.7, xs[-1] + 0.6); ax.set_ylim(19.5, 34.2)
ax.set_yticks(range(20, 35, 2)); ax.set_yticklabels([f'0.954{v:02d}' for v in range(20, 35, 2)])
ax.set_ylabel('public LB'); ax.grid(True, axis='y', color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[plt.Line2D([], [], color=BLUE, marker='o', lw=2.5, ms=8, mec=SURFACE, label='our ladder on the board'),
                   plt.Line2D([], [], color=MUTED, marker='o', mfc=SURFACE, mew=1.8, lw=0, ms=10, label='forecast made before submitting'),
                   plt.Line2D([], [], color=ORANGE, ls=(0, (4, 3)), lw=1, label='top public notebooks, 17 Nov afternoon')],
          loc='lower right', frameon=False, fontsize=9.5)
head(ax, 'Past the public top, one forecast at a time', 'the last five files of the ladder, each new step forecast first')

# right: forecast against board, this round
ax = fig.add_axes([0.57, 0.08, 0.40, 0.55])
lo, hi = -13.5, 8.5
ax.plot([lo, hi], [lo, hi], color=AXIS, lw=1, zorder=1)
ax.axhline(0, color=GRID, lw=1, zorder=1); ax.axvline(0, color=GRID, lw=1, zorder=1)
ax.scatter([b for b, _ in OTHERS], [f for _, f in OTHERS], s=58, color=MUTED, edgecolor=SURFACE, linewidth=1.2, zorder=3, alpha=0.9)
LAB = {'step 10, balanced': ((9, -6), 'left'), 'step 10': ((-10, 4), 'right'), 'step 11': ((10, -5), 'left'), 'step 11 + 20% BhuvanNR file': None,
       'step 12': ((10, 5), 'left'), 'step 13': ((-12, -2), 'right'), 'step 13, larger': None, 'step 13 + CV step': ((-10, 2), 'right')}
for name, (b, f) in OURS.items():
    this = name == 'step 13'
    ax.plot([b], [f], marker='o', ms=12 if this else 10, color=BLUE, mec=INK if this else SURFACE, mew=1.6, zorder=6)
    if LAB[name] is None: continue
    off, ha = LAB[name]
    ax.annotate(name, (b, f), xytext=off, textcoords='offset points', fontsize=9.5 if this else 9, color=INK if this else INK2, ha=ha, va='center',
                fontweight='bold' if this else 'normal')
ax.text(-10.6, -10.1, 'forecast = board', fontsize=9, color=MUTED, ha='center', va='bottom', rotation=42)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('public LB change on the board, points'); ax.set_ylabel('forecast, points')
ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[plt.Line2D([], [], color=BLUE, marker='o', lw=0, ms=9, label='our eight probes (forecast first)'),
                   plt.Line2D([], [], color=MUTED, marker='o', lw=0, ms=8, label='20 files other authors scored (left out in turn)')],
          loc='lower right', frameon=False, fontsize=9.5)
head(ax, 'Forecast against board, 16 and 17 Nov', f'20 public files: forecast error {RMSE16:.1f} points with each file left out')
plt.show()

# S6E11 LM Ladder: ladder meets stack

**Public LB 0.95432**: 2 points above version 9 and 11 above version 6 (1 point = 0.00001).

**New in version 10.** Areeb Ali split our last step in two: the part that lies along a direction already on the board (BhuvanNR's
file minus our 0.95427 file, a third of it) and the rest, `r`. Half a step more of `r` scored 0.95431. His two scores are two more
equations (105 scored files now). Step 13 takes a member step chosen with them and a full `r`: forecast +2.57, written down first;
the board gave +2.

**Version 9** added step 12 (0.95430) after two overnight 0.95428 files.
**Version 8** brought alexchilton's shallow-tree stack into the ladder (step 10, 0.95426) and took one step along the move Asterios
Panoras found (step 11, 0.95427). Every probe is listed below with the forecast written down before it was submitted.

The notebook writes six files. `submission.csv` is the 0.95432 file (step 13), built on the version 5 file, which is tuned on the
public rows. `submission_cv.csv` (0.95379) is the CV-backed file of version 7: the best out-of-fold AUC we have, and the one we
would take for the private rows. `submission_step12.csv` (0.95430) is version 9, `submission_step11.csv` (0.95427) and `submission_step10.csv` (0.95426) are the steps of version 8,
and `submission_bold.csv` is version 4.

This notebook keeps one ladder and adds a rung in each version. Every probe is listed, including the ones that lost.

| version | public LB | what it added |
|---|---|---|
| v1 | 0.95346 | a step towards a target-first LM we retrained ourselves (seed 7) |
| v2 | 0.95354 | that LM on five folds, two LMs at equal weight, and a second GPT-2 seed |
| v3 | 0.95370 | moved to Aman Atar's public 0.95370 file and measured it on CV: public and CV pulled in opposite directions |
| v4 | 0.95386 | a way to pick moves that raise both, checked on the board three times |
| v5 | 0.95417 | the public model in the lead: ten probes with their forecasts, seven up, one of them up on CV as well |
| v6 | 0.95421 | three more scored public files in the model, and a step that raises the board (+4) and nested CV (+7.7, all five folds) |
| v7 | 0.95421 | a new CV-backed file (OOF 0.95266): language-model weight moved to the GPT-2 seeds, +7.3 nested; and what a stronger model of the same family is worth to the blend |
| v8 | 0.95427 | sixteen files scored by other authors as new equations; alexchilton's shallow-tree stack joins the ladder (step 10, 0.95426), one step along Asterios Panoras' move (step 11, 0.95427) |
| v9 | 0.95430 | two overnight 0.95428 files as equations 98 and 99; step 12: a member step plus shares of the two directions they used |
| **v10** | **0.95432** | **Areeb Ali's split of our last step as two more equations; step 13: a member step plus the new part of step 12** |

## Version 10: the new part of the last step

**1. A split of our own step.** Areeb Ali's *Split, Then Push* (0.95431) writes our step 12 as `d = step 12 - step 11` and the
known direction `e = B - step 11` (BhuvanNR's file B), takes `d = 0.331 e + r` by least squares on the test rows, and adds `0.5 r`.
The same author scored `0.5 step 12 + 0.5 A` (A: Asterios Panoras' 0.95428 file) at 0.95429. We rebuilt both files exactly; with
them the public model holds 105 scored files.

**2. Step 13.** From the 0.95430 file the refitted model finds a member step over the same 88 members (+1.8 forecast on its own),
and a full `r` on top (+1.5 on its own). Together they forecast **+2.57**, with 98% of the move inside directions scored files have
tested. The board gave **+2: 0.95432**. A larger version of the same step (1.5 x the member step and a quarter of `A - O`) also
scored 0.95432.

**3. Where it stops.** Refitted with those two scores, the model sees no further gain along these directions (+0.02 at best). The
moves it does find from here raise nested CV (up to +8 in all five folds) at a forecast of about -0.3. One of them was probed and
scored 0.95431, a point down, so this version keeps step 13.

**4. What it costs on out-of-fold rows.** On the replayed OOF the member step is -7.2 and step 13 as a whole -16.5 (0 of 5 folds up),
-31.5 from the version 6 file. `submission_cv.csv` stays the CV-backed file.

| step | starts from | forecast | public LB | OOF change (replay) |
|---|---|---|---|---|
| step 12 (version 9) | 0.95427 | +3.05 | 0.95430 (+3) | -12.4 |
| **step 13 (this version)** | 0.95430 | **+2.57** | **0.95432 (+2)** | -16.5 |

## Version 9: one more step past two new 0.95428 files

**1. Two new equations.** Areeb Ali's *Two Ladders, One Average* (0.95428) is the plain logit-rank average of our 0.95427 file (G)
and BhuvanNR's step-10 file (B). Asterios Panoras' 0.95428 adds `0.25 (B - B420) + 0.5 (A - O)` to that average. We rebuilt the
average exactly (identical ranks). With both scores in, the model forecasts each of them to within 0.4 points when it is left out.

**2. Mixes alone stop at 0.95428.** From our 0.95427 file, every mix of `G`, `B`, `A - O` and `B - B420` we tried forecasts at most
+1.0. The board agrees: two such mixes scored 0.95428.

**3. Step 12 adds a member step.** Same question as before, asked from the 0.95427 file with 99 equations: maximise
`CV change + mu x forecast` over the 88 members (87 + the stack). The member step alone forecasts +2.1; with
`0.3 (B - G) + 0.5 (A - O)` on top it forecasts **+3.05**, and 98% of the combined move lies inside directions scored files
have tested. The board gave **+3: 0.95430**. The largest member moves: the stack +0.12, a one-hot GLM +0.05, masayakawamata's
GPT-2 -0.05, Sameer Shaikh's GLM -0.04.

**4. What it costs on out-of-fold rows.** On the replayed OOF the member step is -5.7 and step 12 as a whole -12.4 (0 of 5 folds
up): a step the board pays for and CV does not. `submission_cv.csv` stays the CV-backed file.

| step | starts from | forecast | public LB | OOF change (replay) |
|---|---|---|---|---|
| step 11 (version 8) | 0.95426 | +1.0 | 0.95427 (+1) | -5.7 |
| **step 12 (this version)** | 0.95427 | **+3.05** | **0.95430 (+3)** | -12.4 |

## Version 8: ladder meets stack

**1. Sixteen new equations.** On 16 Nov four notebooks scored files near our 0.95421 file, and each one states its scores:
Areeb Ali's *Two Families, One Blend* (our ladder file X mixed with alexchilton's shallow-tree stack S, six shares from 6% to 35%,
and X doubled), his *Steer, Then Mix* (eight probes around an 80/20 mix of Asterios Panoras' 0.95422 file and ours), Asterios
Panoras' *This Score Is Not Real* (0.95424) and BhuvanNR's *Fable & Two Families* (0.95420 and 0.95426). We rebuilt every file
whose recipe is published and checked three of them against the authors' outputs (identical ranks). Added to the public model,
they bring it to 95 scored files. With each of the 16 left out in turn, the model forecasts it to 0.5 points (root mean square).

**2. Step 10: the stack as a member.** The 87 members of earlier versions could not reach what the board rewarded in those files:
the stack itself. alexchilton's stack file has no out-of-fold predictions, but it is a linear blend of public models, and a fit on
the test rows explains 99.87% of it with our 87 members; the same weights give its out-of-fold side. With the stack as member 88,
the step that maximises `CV change + mu x forecast` from the 0.95421 file:

| step | starts from | members | CV change (nested) | folds up | forecast | public LB |
|---|---|---|---|---|---|---|
| step 10, balanced | 0.95421 | 87 + stack | +5.4 | 4/5 | +5.0 | 0.95424 (+3) |
| **step 10** | 0.95421 | 87 + stack | +0.5 | 3/5 | **+6.4** | **0.95426 (+5)** |

Weight 0.48 goes to the stack, and smaller shares to EBM bags and megayak's shallow tree. The nested numbers were measured offline
with alexchilton's 38 public models in the replay as well; the cell *Version 8* below uses the 87 members only and prints its own
in-sample numbers. The stack's weights were fitted on the labelled rows, so its replay flatters the CV column a little; read it as
"about level".

**3. Step 11: one step along Asterios Panoras' move.** His 0.95422 file (A) was steered from Areeb Ali's 0.95421 file (O) by a
model of the board; Areeb Ali then found that the board kept paying along `A - O`. With step 10 in the model, the forecast for
`step 10 + 1.0 (A - O)` is +1.0, and the board gave +1: **0.95427**. On the replayed out-of-fold rows the move costs 5.7 points; from
the version 6 file the two steps together are -2.5.

**4. Where it stops.** From step 11 every further move the model can see trades CV for the board at about 3 to 5 points per point.
One such probe (20% of BhuvanNR's 0.95426 file mixed in) scored 0.95428 and is not used: it needs a file nobody has pinned, and it
gives up 6 more points of out-of-fold AUC.

## Version 7: what the blend wants, and a new CV-backed file

`submission.csv` did not change in version 7 (0.95421). The CV-backed file `submission_cv.csv` does. It comes out of two
measurements on the CV-led file of version 5, both computed in the cell marked *Version 7* below.

**1. A stronger model of the same family buys almost nothing.** The 87 members hold 23 pairs in which one model is a stronger
version of the other: one seed against a bag of seeds, three bags against five, a different tree size. Move weight from the weaker
one to the stronger one. For the 11 pairs where the stronger model is at least 5 points better alone, it is better by 38 points
on average, and the blend gains 0.15. The blend keeps 0.4% of the difference. A bigger bag of a model the blend already holds
does not pay for its compute here.

**2. Among the language models, the blend does not ask for the best one.** Move 2% of the file towards each member in turn.
Across all 87 members the gain follows the member's own score (rank correlation +0.84): weak members are not wanted. The four
target-first language models break that pattern. The two GPT-2 seeds score 0.9500 and 0.9501 alone and top the whole list
(+1.1 and +1.3, up in all five folds). megayak's compact LM scores higher alone (0.9504), and the blend wants less of it (-0.5).

**3. So the step moves weight inside the language-model family**, from megayak's LM to the average of the two GPT-2 seeds.
With 5% moved the file gains 3.8 points, with 10% 6.5, with 15% 8.2 (up in all five folds), with 20% 8.7 (four folds). With the
size chosen on four folds and scored on the fifth it is +7.3, up in four folds of five. We take 15%, the largest gain that is up
in every fold.

| file | OOF AUC | public LB |
|---|---|---|
| the CV-led step of version 5 (the previous `submission_cv.csv`) | 0.952583 | 0.95387 |
| **the same with 15% of megayak's LM moved to the GPT-2 seeds (`submission_cv.csv` now)** | **0.952664** | 0.95379 |

On the board the new file is 8 points below the previous one; the public model had forecast -6.6 before it was submitted. The
60,000 public rows lean against this direction, and the 700,000 labelled rows lean for it. For the private rows we go with the
larger sample.

One detail of the test side: the file uses the five-fold test predictions of GPT-2 seed 2026
([dataset](https://www.kaggle.com/datasets/goodpjw2008/s6e11-target-first-gpt2-seed-2026)), where the bundle carried the fold-0
model's. Out-of-fold numbers are not affected.

## Version 6: a step the board and CV both reward

Version 5 ended with the slope spent: the model's best forecast for a further move was smaller than its own error. Since then:

**1. Three more files were scored in public.** BhuvanNR rebuilt the public model in the open and took one more step from our
0.95417 file ([0.95419](https://www.kaggle.com/code/bhuvannr2006/s6e11-lm-ladder-forecast-the-board-lb-0-95417)). Areeb Ali mixed
10% of alexchilton's shallow-tree stack into that file
([Two Families, One Blend](https://www.kaggle.com/code/areebbb/s6e11-two-families-one-blend-lb-0-95421), 0.95421). Asterios Panoras
added one more probe and measured what each step costs on out-of-fold rows
([This Score Is Not Real](https://www.kaggle.com/code/asteriospanoras/s6e11-lb-0-95422-this-score-is-not-real), 0.95422).
Each stated score is one more equation about the public rows. The model had 69 scored files at the end of version 5 and 75 now
(these three, and three of ours).

**2. With those equations the model can rank directions it was blind to.** We asked the question of version 4 again: among moves
of the same 87 members, which one raises `CV change + mu x forecast` with nested CV up in all five folds? Both steps start from the
version 5 file.

| step | starts from | led by | CV change (nested) | folds up | forecast | public LB |
|---|---|---|---|---|---|---|
| step 8 | 0.95417 | CV and forecast | +9.6 | 5/5 | +5.3 | 0.95418 (+1) |
| **step 9 (this version)** | 0.95417 | **CV and forecast** | **+7.7** | **5/5** | **+3.4** | **0.95421 (+4)** |

* **Step 8 raised both, by less than forecast.** A quarter of its move lay outside every direction a scored file had tested. The
  board gave +1 against a forecast of +5.3, and the miss became equation 76.
* **Step 9 is the same question asked again with that equation.** 98% of its move lies inside scored directions. Forecast +3.4,
  board +4. On out-of-fold rows it is +7.7 points nested (weights chosen on four folds, scored on the fifth; up in all five) and
  +12.7 in-sample.
* **What moved.** Weight went to megayak's shallow tree (+0.21) and to EBM bags, away from single EBM and one-hot GLM models:
  the direction Areeb Ali's mix with a shallow-tree stack had pointed at.
* **Where it stops.** Refitted with the 0.95421 score, the model finds no further move that raises both. Each further point on the
  board costs about 5 points of nested CV, so the ladder stops here for now.

## Version 5: what happened since version 4

**1. Areeb Ali extrapolated the version 4 mixes.** His notebook
[Bold, Then Bolder](https://www.kaggle.com/code/areebbb/s6e11-bold-then-bolder-lb-0-95388) scored eight files of the form
`a x bold + b x balanced + c x careful`. They landed between 0.95385 and 0.95388; the best, `bold + 0.5 (bold - balanced)`,
is 0.95388. On out-of-fold rows that step is up about 4 points, in all five folds (computed below).

**2. His scores were a free test of our public model.** The version 4 model was fitted before his scores existed. Its forecasts for
his eight files miss by 1.6 points (root mean square). Saying "no change" for every file misses by 1.1. Along directions that
out-of-fold AUC supports, the top is flat and the model cannot rank files.

**3. Ten steps of ours.** Same method as version 4 (below): maximise `CV change + mu x forecast`. After each score the public
model was refitted with it. Changes are from the file a step starts from; forecasts were written down before submitting.

| step | starts from | led by | CV change (nested) | folds up | forecast | public LB |
|---|---|---|---|---|---|---|
| joint | 0.95388 | forecast and CV | +3.3 | 3/5 | +3.1 | 0.95388 (0) |
| CV-led | 0.95388 | CV | +7.3 | 5/5 | -0.3 | 0.95387 (-1) |
| step 1 | 0.95388 | forecast | -1.1 | 2/5 | +1.0 | 0.95390 (+2) |
| step 2, half size | 0.95390 | forecast | -8.6 | 0/5 | +3.7 | 0.95393 (+3) |
| step 2 | 0.95390 | forecast | -18.4 | 0/5 | +6.5 | 0.95396 (+6) |
| step 3 | 0.95396 | forecast | -29.2 | 0/5 | +4.9 | 0.95404 (+8) |
| step 4 | 0.95404 | forecast | -18.6 | 0/5 | +5.1 | 0.95408 (+4) |
| step 5 (not kept) | 0.95408 | forecast | -33.3 | 0/5 | +1.4 | 0.95407 (-1) |
| step 6 | 0.95408 | CV, where the model is blind | +16.0 | 5/5 | +0.1 | 0.95412 (+4) |
| step 7 (version 5) | 0.95412 | forecast | -19.0 | 0/5 | +4.2 | 0.95417 (+5) |

* **With CV in the lead, nothing moved.** The step with the best CV lost a point. That is the flat top again.
* **With the forecast in the lead, four steps paid on the board.** Four steps, four gains, 20 points in all. Out-of-fold AUC
  went the other way at each of them (the CV column above).
* **Then the slope ran out.** Refitted with the 0.95408 score, the model's best forecast for any further move of these 87
  members was +1.4 points, at a CV cost of 33. We submitted it the next morning: 0.95407, one point down. Along the directions
  tested so far, the slope was spent.
* **Step 6 went back towards CV where the board is blind.** By then the file was 50 points down on CV. Step 6 puts CV in the
  lead again (+16 nested, all five folds) and 30% of its move lies outside the directions any scored file had tested. For that
  part the best guess of the public response is the CV response itself: the public rows are a sample of the same data. What the
  model knew gave a forecast of +0.1; with that guess, +4.8. The board gave +4. It is the one step here that raised both.
* **Step 7 was led by the forecast again:** +4.2 forecast, +5 on the board, and 19 points of CV given back. Aman Atar's new
  public file (0.95390) was added to the public model before steps 6 and 7 as one more scored file.
* **What the model is doing.** Each scored file is one linear equation about the 60,000 public labels (formula below). By the last
  step there were 69 equations. They do not identify the labels, but they identify the directions the scored files span, and a
  move along the estimated direction raises the public score almost by construction. This is leaderboard probing written as
  linear algebra. It needs no new model.
* **What moved.** Weight went from megayak's shallow tree and several single digit models to EBM bags and other digit-lead bags.
  Nothing on out-of-fold rows asks for that.

**Which file to use.** `submission.csv` is the 0.95432 file (step 13). `submission_cv.csv` is the CV-backed file of version 7
(0.95379): the highest out-of-fold AUC we have, which is the better guide to the private rows. `submission_bold.csv` is version 4.

## Version 4: the problem version 3 left open

Version 3 showed a trade. Mixing a CV-backed stack into the 0.95370 file raised out-of-fold AUC and lowered the public score.
Weights chosen on out-of-fold AUC alone would have gained 39 points of CV and, by the forecast below, lost 22 on the board.

## How the mixes are picked

1. **A public score moves linearly with ranks.** To first order,

   `AUC(file) - AUC(reference) = mean over rows of (y - p(u)) * (u_file - u) / (pi (1 - pi))`

   where `u` is a row's rank fraction in the reference file, `p(u)` the fraud rate at that rank and `pi` the overall fraud rate.
   The only unknown is one number per public row, `y - p`.
2. **Scored files pin those unknowns down.** Each file we have a public score for is one linear equation about them. With 44 of our
   own scored files, a ridge fit on rank differences (rows weighted by `sqrt(p (1 - p))`, plus one curvature term) forecasts a
   left-out file's public score to **1.4 points** (root mean square; 4.2 with the curvature term alone), over a range of
   50 points.
3. **Forecast and CV are maximised together.** For moves towards 87 members (the 16 public models of the bundle, our wide LM stack,
   70 models of [our OOF bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank)) we take the mix that maximises
   `CV change + mu * forecast public change`, with a penalty on the part of a move the scored files say nothing about.
   The CV number is then measured again with the weights refitted on four folds and scored on the fifth.

## What the board said in version 4

All forecasts were written down before submitting. Changes are from the 0.95370 file.

| mix | CV change (nested) | folds up | forecast | public LB |
|---|---|---|---|---|
| careful | +22.9 | 5/5 | +3 | 0.95376 (+6) |
| balanced | +14.3 | 5/5 | +8 | 0.95380 (+10) |
| bold (version 4) | +19.5 | 5/5 | +11 | 0.95386 (+16) |
| bold + one more step (adds Strike 10 minus Strike 9) | +2.7 on top of bold | 3/5 | +4 on top of bold | 0.95385 (-1) |

* **The three mixes with strong CV support beat their forecasts** by 2 to 5 points. The forecast only covers the part of a move
  that earlier files span; the rest of the move is real signal the model cannot see, and the board paid for it.
* **The step with weak CV support missed.** It was mostly a public-ladder direction, CV was up in 3 folds of 5, and the forecast
  was 5 points too high. A forecast without CV behind it is not enough.
* The largest moves of the bold mix: megayak's shallow tree +0.28, daoviet's GP+DVAE +0.14, megayak's LM -0.12, sadam's XGB -0.10,
  masaya's GPT-2 -0.07 (shares of the file moved towards or away from each model). The file had the shallow tree at a negative
  weight; undoing part of that is where most of the gain is.

Read the limits below before using these numbers.

## The analysis, step by step

The weights of the three mixes (`RECIPE_JSON`), of our steps (`STEP_JSON`) and of step 10 (`STEP10_JSON`) are defined at the top of
the next cell; everything below is computed from public inputs.

In [ ]:
# Weights of the three scored mixes (member -> share moved towards it). s: a public model in the bundle, b: a model of our OOF bank.
RECIPE_JSON = '''{
"bold": {"s:megayak_5f_lm": -0.12205919443860914, "s:masaya_gpt2": -0.06997547490341371, "s:lm7_5fold": -0.040956430057266076, "s:gpt2_seed2026": -0.012508511564926699, "s:tabpfn_slimypunk_tp1": -0.039316699160350764, "s:sadam_tabpfn_100k_tp3": -0.061677762652753675, "s:yekenot_realmlp": -0.014637185195237004, "s:alex_glm_gen": 0.004537993603062768, "s:daoviet_gpdvae": 0.1359104270990657, "s:gbn_bag5_w512": 0.015432253915041055, "s:sameer_glm10f": 0.06272836350651828, "s:sadam_v6_gb_xgb": -0.10228295915095288, "s:sadam_v6_gb_lgb": 0.06627713517089104, "s:sadam_v4_cat": 0.03126101632182668, "s:gb_bag5": 0.008855983394809901, "s:megayak_shallow_tree": 0.28129985621486264, "wide": -0.034852407295305754, "b:ebm_all_bag": -0.050611080400723436, "b:ebm_dig": -0.016392951045769617, "b:ebm_dig_bag": -0.021971929033966105, "b:ebm_dig_r7": 0.03715898221284049, "b:ebm_diglead_bag": -0.011280640758210015, "b:ebm_diglead_i25": -0.01760600397929515, "b:ebm_diglead_r11": 0.020606132176681696, "b:ebm_diglead_r13": 0.031952532024431754, "b:ebm_i25_bag": 0.019937230510505454, "b:ebm_i25_bag4": 0.04011014273129195, "b:ebm_lead4_bag": -0.016285931935351022, "b:gb_bag10": -0.017390237028207023, "b:gb_bag5": 0.026782938331671603, "b:gb_l3": -0.06609634322241796, "b:gb_ohk": -0.02882833922678926, "b:gbc_bag3": -0.012871505523924395, "b:gbe_bag3": 0.012674094986094425, "b:gbn_bag5_w128": -0.029120675300013294, "b:gbn_bag5_w512": 0.04504705483635584, "b:gbr_bag3": -0.005251606394839109, "b:gbx_bag3": -0.01901561844474828, "b:gbx_bag5": 0.006799891770028022, "b:glm_dig": 0.003088377583964293, "b:glm_dl": -0.006428517981988452, "b:glm_dl_bag": 0.01502056793927036, "b:glm_dl_s50": -0.0501984153162254, "b:glm_dl_s50_bag": -0.028630562092033966, "b:glm_ohk": -0.017070904616440657, "b:glm_ohk2": -0.004903067440865727, "b:glm_ohk2_bag": -0.029877217995289047, "b:glm_ohk_c01": 0.0003448885987200072, "b:glm_ohkl": 0.006673473442067076, "b:glm_ohkl_bag": 0.01766365615186144, "b:glm_sb": -0.01145379830324528, "b:glm_te": -0.004792258197303907, "b:lgb_dig_l4": 0.0366960138715965, "b:lgb_dig_l7": 0.01949687340453925, "b:lgb_dig_s100": -0.051481356730372616, "b:lgb_dig_s100_bag": -7.44946546564366e-05, "b:lgb_dig_s100_bag5": 0.016566984017469158, "b:lgb_dig_s100_l3": 0.039879653631481535, "b:lgb_dig_s100_l3_bag": -0.014332571073509623, "b:lgb_diglead_bag": -0.00442949281469811, "b:lgb_diglead_bag5": -0.05069793900237127, "b:lgb_diglead_l2": -0.0434403013635199, "b:lgb_diglead_l3": 0.051384825410253424, "b:lgb_diglead_l3_bag": -0.014548087315257047, "b:lgb_diglead_l4": 0.005059024340666011, "b:lgb_dk_l4": 0.010395272568768744, "b:lgb_raw_l31": 6.272373140902098e-05, "b:lgb_raw_l4": 0.006062225361634528, "b:lgb_raw_l4_cnt": -0.060074422527671846, "b:lgb_rawdig_l4": 0.012821826099948688, "b:lgb_te_l4": 0.026453964192257394, "b:lgb_te_l4_s100": 0.007166689614341422, "b:lgb_te_l4_s300": 0.009325395175060891, "b:lgb_te_l4_s5": 0.00877347694863135, "b:realmlp_allcat": -0.010498730399732318, "b:realmlp_cnt": 0.00748969325179405, "b:realmlp_dig": -0.026045115561739243, "b:realmlp_dig_s100": 0.01482032375010226, "b:realmlp_dig_s2": 0.012780340101831603, "b:realmlp_rawdig": -0.029224792823957325, "b:realmlp_te": 0.011902655678168103, "b:realmlp_te_cnt": -0.0016811066269879565, "b:realmlp_te_s2": -0.00033527900132535184, "b:realmlp_v1": 0.02099235698670545, "b:xgb_dig_d2": -0.027214651902440812, "b:xgb_raw_d3": 0.013799277379315805, "b:xgb_te_d2": -0.012792356993134408},
"balanced": {"s:megayak_5f_lm": -0.0659946888896202, "s:masaya_gpt2": -0.07115470402589986, "s:lm7_5fold": -0.029998755707584368, "s:gpt2_seed2026": 0.005917733255070143, "s:tabpfn_slimypunk_tp1": -0.016754854043217735, "s:sadam_tabpfn_100k_tp3": -0.02209424107030813, "s:yekenot_realmlp": -0.007469976912482503, "s:alex_glm_gen": 0.004294670805904037, "s:daoviet_gpdvae": 0.07254584097984837, "s:gbn_bag5_w512": -0.002874661821916413, "s:sameer_glm10f": 0.006253891711281581, "s:sadam_v6_gb_xgb": -0.035779752095592386, "s:sadam_v6_gb_lgb": 0.014771844136765163, "s:sadam_v4_cat": 0.005442134718824526, "s:gb_bag5": 0.02197944620878155, "s:megayak_shallow_tree": 0.10861837977873254, "wide": 0.008857483348604336, "b:ebm_all_bag": -0.035648523256013816, "b:ebm_dig": 0.0021571996755421373, "b:ebm_dig_bag": -0.00870932789686656, "b:ebm_dig_r7": 0.014675020864131676, "b:ebm_diglead_bag": -0.011242137972560769, "b:ebm_diglead_i25": -0.011799461220371074, "b:ebm_diglead_r11": 0.018395467162953583, "b:ebm_diglead_r13": 0.019913528869070343, "b:ebm_i25_bag": 0.014300715796457614, "b:ebm_i25_bag4": 0.014776764302437287, "b:ebm_lead4_bag": -0.010845849051441005, "b:gb_bag10": -0.01586540001387272, "b:gb_bag5": 0.013795945657627443, "b:gb_l3": -0.022653116683931083, "b:gb_ohk": -0.005825205521352985, "b:gbc_bag3": -0.006125060624374499, "b:gbe_bag3": 0.004479022738251753, "b:gbn_bag5_w128": -0.008463251429546712, "b:gbn_bag5_w512": 0.02476864383685032, "b:gbr_bag3": -9.909508786856773e-06, "b:gbx_bag3": -0.006276689543614139, "b:gbx_bag5": 0.005886331747934474, "b:glm_dig": 0.0005311280711308347, "b:glm_dl": -0.0059032327276077766, "b:glm_dl_bag": 0.014006601323714443, "b:glm_dl_s50": -0.022355265176809174, "b:glm_dl_s50_bag": 0.0006990668756181338, "b:glm_ohk": -0.005312350261809259, "b:glm_ohk2": 0.00678460176455991, "b:glm_ohk2_bag": -0.014305563020749566, "b:glm_ohk_c01": 0.002124044507600178, "b:glm_ohkl": -0.0018632849202664824, "b:glm_ohkl_bag": 0.009617257091917682, "b:glm_sb": -0.0032167242579132175, "b:glm_te": -0.0024289780895983912, "b:lgb_dig_l4": 0.010390240631489167, "b:lgb_dig_l7": 0.006166614002474661, "b:lgb_dig_s100": -0.016328847468105947, "b:lgb_dig_s100_bag": 0.006054175271612747, "b:lgb_dig_s100_bag5": 0.0032893020819732264, "b:lgb_dig_s100_l3": 0.01156644449639154, "b:lgb_dig_s100_l3_bag": -0.0072697486729145595, "b:lgb_diglead_bag": 0.003752570618535255, "b:lgb_diglead_bag5": -0.01387844815389441, "b:lgb_diglead_l2": -0.008722162257095078, "b:lgb_diglead_l3": 0.01340449460759475, "b:lgb_diglead_l3_bag": -0.009648198615136732, "b:lgb_diglead_l4": 0.0023017753746317295, "b:lgb_dk_l4": 0.00404320185129622, "b:lgb_raw_l31": 0.0004917472972874557, "b:lgb_raw_l4": -0.00011914588071201624, "b:lgb_raw_l4_cnt": -0.015582993656188236, "b:lgb_rawdig_l4": 0.001729542007007652, "b:lgb_te_l4": 0.008969005313344148, "b:lgb_te_l4_s100": 3.676618403346857e-05, "b:lgb_te_l4_s300": 0.0022842969787489017, "b:lgb_te_l4_s5": 0.002931297790132998, "b:realmlp_allcat": -0.0026909874980677114, "b:realmlp_cnt": 0.0021648947039792175, "b:realmlp_dig": -0.009016404783169642, "b:realmlp_dig_s100": 0.005402396661241892, "b:realmlp_dig_s2": 0.0022649717220251738, "b:realmlp_rawdig": -0.007755994202623359, "b:realmlp_te": 0.0032482793804713803, "b:realmlp_te_cnt": 0.000847393585301589, "b:realmlp_te_s2": 0.0004964361210193442, "b:realmlp_v1": 0.0064450941750321756, "b:xgb_dig_d2": -0.001615855509527677, "b:xgb_raw_d3": 0.00433269057718474, "b:xgb_te_d2": -0.0033627181678075714},
"careful": {"s:megayak_5f_lm": -0.08450474886996073, "s:masaya_gpt2": -0.04902561986411483, "s:lm7_5fold": -0.016891411336523863, "s:gpt2_seed2026": 0.014547738041531408, "s:tabpfn_slimypunk_tp1": -0.0123457470481866, "s:sadam_tabpfn_100k_tp3": -0.015910384586476225, "s:yekenot_realmlp": -0.003253325303861341, "s:alex_glm_gen": 0.002612752988313417, "s:daoviet_gpdvae": 0.04685747302679586, "s:gbn_bag5_w512": 0.0043515522989154235, "s:sameer_glm10f": 0.015478953160303576, "s:sadam_v6_gb_xgb": -0.028037375140335462, "s:sadam_v6_gb_lgb": 0.019533386719025193, "s:sadam_v4_cat": 0.00589878484038633, "s:gb_bag5": 0.02217844040888381, "s:megayak_shallow_tree": 0.10122717023890326, "wide": 0.08081797187087163, "b:ebm_all_bag": -0.03128848896508937, "b:ebm_dig": 0.002420259254897357, "b:ebm_dig_bag": -0.006646622819934664, "b:ebm_dig_r7": 0.010794306141448952, "b:ebm_diglead_bag": -0.0117042245417344, "b:ebm_diglead_i25": -0.008918615668229131, "b:ebm_diglead_r11": 0.017495433781515475, "b:ebm_diglead_r13": 0.019436209350096583, "b:ebm_i25_bag": 0.008522596080523918, "b:ebm_i25_bag4": 0.016476990933893144, "b:ebm_lead4_bag": -0.01130787311582097, "b:gb_bag10": -0.013583645440115318, "b:gb_bag5": 0.016684891246275074, "b:gb_l3": -0.019083828087590973, "b:gb_ohk": -0.006177878317621568, "b:gbc_bag3": -0.007269120619879105, "b:gbe_bag3": 0.004681311487245302, "b:gbn_bag5_w128": -0.005040768827170892, "b:gbn_bag5_w512": 0.023637688632069328, "b:gbr_bag3": -0.00024830818253431273, "b:gbx_bag3": -0.006400911249302103, "b:gbx_bag5": 0.0036937689629500865, "b:glm_dig": -3.7306124542630744e-05, "b:glm_dl": -0.007156160004066149, "b:glm_dl_bag": 0.008441014574504154, "b:glm_dl_s50": -0.01334934207158471, "b:glm_dl_s50_bag": 0.001844211766019183, "b:glm_ohk": -0.00562390612264448, "b:glm_ohk2": 0.008907953791620765, "b:glm_ohk2_bag": -0.015300133811741116, "b:glm_ohk_c01": -0.005892186175925242, "b:glm_ohkl": -0.00010117004854648993, "b:glm_ohkl_bag": 0.008342574170709143, "b:glm_sb": -0.003455433101783884, "b:glm_te": -0.0018554925449663707, "b:lgb_dig_l4": 0.012788668778136636, "b:lgb_dig_l7": 0.006133544143597403, "b:lgb_dig_s100": -0.01581131868317444, "b:lgb_dig_s100_bag": 0.005976683240784171, "b:lgb_dig_s100_bag5": 0.003272572328457066, "b:lgb_dig_s100_l3": 0.011358913543846424, "b:lgb_dig_s100_l3_bag": -0.007671455946372908, "b:lgb_diglead_bag": 0.0026446221720072436, "b:lgb_diglead_bag5": -0.014415491763205732, "b:lgb_diglead_l2": -0.009879458752685616, "b:lgb_diglead_l3": 0.015081200121801388, "b:lgb_diglead_l3_bag": -0.01231464394864841, "b:lgb_diglead_l4": 0.0027104104670708385, "b:lgb_dk_l4": 0.002097897271309318, "b:lgb_raw_l31": 0.0010331280960220208, "b:lgb_raw_l4": -0.0013378069288362123, "b:lgb_raw_l4_cnt": -0.013507197348019013, "b:lgb_rawdig_l4": 0.0028805514532224763, "b:lgb_te_l4": 0.008125965905514013, "b:lgb_te_l4_s100": 0.0030857860977705903, "b:lgb_te_l4_s300": 0.0022465238467168505, "b:lgb_te_l4_s5": -0.0007517378836866291, "b:realmlp_allcat": -0.00013519997284003595, "b:realmlp_cnt": 0.0031564480735426013, "b:realmlp_dig": -0.009290279722601536, "b:realmlp_dig_s100": 0.005582584293801132, "b:realmlp_dig_s2": 0.0018383277487227473, "b:realmlp_rawdig": -0.009112515110809292, "b:realmlp_te": 0.003079185734258433, "b:realmlp_te_cnt": -0.00037244665110067394, "b:realmlp_te_s2": -0.0021716481344977163, "b:realmlp_v1": 0.005001493601478614, "b:xgb_dig_d2": -0.004210955412618814, "b:xgb_raw_d3": 0.004254033686957146, "b:xgb_te_d2": -0.0023974804930285173}
}'''
# Our 44 scored files (not shipped): public change against the 0.95370 file, and the model's forecast with that file left out.
LOO_ACTUAL = [-103, -103, -109, -85, -87, -96, -88, -89, -72, -43, -39, -37, -30, -29, -29, -29, -29, -28, -29, -27, -25, -41, -46, -27, -26, -24, -23, -22, -18, -18, -19, -16, -16, -15, -1, -2, -2, -8, 0, -40, -30, -36, -33, -31]
LOO_PRED = [-100.2, -103.1, -112.6, -85.4, -89.5, -93.6, -87.8, -87.4, -71.4, -42.5, -36.9, -36.3, -31.5, -30.0, -29.0, -29.3, -29.3, -28.6, -29.2, -28.5, -25.6, -39.2, -49.3, -26.6, -25.5, -22.2, -20.8, -21.6, -17.4, -17.9, -18.9, -16.9, -16.6, -17.3, 0.6, -1.3, -2.5, -7.9, -0.8, -38.9, -30.2, -35.8, -33.3, -31.5]
# Versions 5 and 6: our twelve steps (which file each starts from is in FROM below), same members.
STEP_JSON = '''{
"joint": {"s:megayak_5f_lm": -0.01717077159073525, "s:masaya_gpt2": 0.0019183243447994148, "s:lm7_5fold": 0.045423022260665136, "s:gpt2_seed2026": -0.00042944375296627196, "s:tabpfn_slimypunk_tp1": -0.00899495055721427, "s:sadam_tabpfn_100k_tp3": -0.029270717860439565, "s:yekenot_realmlp": -0.00234960739485184, "s:alex_glm_gen": -0.02024058123014512, "s:daoviet_gpdvae": 0.03484720930561664, "s:gbn_bag5_w512": 0.043838711156557714, "s:sameer_glm10f": 0.06984336964612282, "s:sadam_v6_gb_xgb": -0.009431902430786612, "s:sadam_v6_gb_lgb": 0.029233124147598495, "s:sadam_v4_cat": 0.01721852315397201, "s:gb_bag5": -0.017019375679840406, "s:megayak_shallow_tree": 0.025418142677893215, "wide": 0.058343674987245644, "b:ebm_all_bag": 0.04569362232028508, "b:ebm_dig": -0.050631120263571604, "b:ebm_dig_bag": 0.03236181449748603, "b:ebm_dig_r7": -0.016790661098617175, "b:ebm_diglead_bag": -0.02032888065903343, "b:ebm_diglead_i25": 0.016077523937003083, "b:ebm_diglead_r11": -0.015234267117895712, "b:ebm_diglead_r13": -0.002050297510730736, "b:ebm_i25_bag": -0.02278394288002672, "b:ebm_i25_bag4": 0.011423726826693454, "b:ebm_lead4_bag": 0.030716217831992645, "b:gb_bag10": 0.0006193248766763178, "b:gb_bag5": 0.01586673751093695, "b:gb_l3": -0.010729425968314792, "b:gb_ohk": -0.024322587520270455, "b:gbc_bag3": 0.010519392802311005, "b:gbe_bag3": 0.008359864921988952, "b:gbn_bag5_w128": 0.002248917510635357, "b:gbn_bag5_w512": 0.018136959102802233, "b:gbr_bag3": -0.008365004797769256, "b:gbx_bag3": 0.006589007276661395, "b:gbx_bag5": -0.03355885475043985, "b:glm_dig": 0.0012813536285928683, "b:glm_dl": -0.001273981615760972, "b:glm_dl_bag": -0.02872643207621259, "b:glm_dl_s50": -0.008710208055072402, "b:glm_dl_s50_bag": -0.021278755306043704, "b:glm_ohk": -0.02123527953417163, "b:glm_ohk2": -0.026270260484603916, "b:glm_ohk2_bag": -0.017602826594117285, "b:glm_ohk_c01": -0.013770304192067154, "b:glm_ohkl": 0.013870805201599829, "b:glm_ohkl_bag": 0.022583324982946185, "b:glm_sb": -0.008448371582005464, "b:glm_te": 0.0016929278315118475, "b:lgb_dig_l4": 0.020239869250518273, "b:lgb_dig_l7": 0.009344260941331638, "b:lgb_dig_s100": -0.014026745793121813, "b:lgb_dig_s100_bag": -0.03897583634328856, "b:lgb_dig_s100_bag5": 0.011268527889825249, "b:lgb_dig_s100_l3": 0.004982508768429224, "b:lgb_dig_s100_l3_bag": 0.019769181140729782, "b:lgb_diglead_bag": -0.04352076765430828, "b:lgb_diglead_bag5": -0.0005833043594665035, "b:lgb_diglead_l2": -0.04413222287065789, "b:lgb_diglead_l3": 0.033124504055349224, "b:lgb_diglead_l3_bag": 0.009405122110509721, "b:lgb_diglead_l4": -0.0018646983927889356, "b:lgb_dk_l4": 0.0005093211910773871, "b:lgb_raw_l31": 0.001729502053405105, "b:lgb_raw_l4": 0.006285249236442414, "b:lgb_raw_l4_cnt": -0.038610384561969635, "b:lgb_rawdig_l4": 0.01664169846460153, "b:lgb_te_l4": 0.024628528641973735, "b:lgb_te_l4_s100": 0.023996989997212066, "b:lgb_te_l4_s300": -0.00853403591839359, "b:lgb_te_l4_s5": 0.0030603366645029534, "b:realmlp_allcat": -0.009471160685104547, "b:realmlp_cnt": 0.0030608219751133265, "b:realmlp_dig": -0.011108471402391146, "b:realmlp_dig_s100": 0.0003058860814721104, "b:realmlp_dig_s2": 0.015435773625366142, "b:realmlp_rawdig": -0.024481211697770403, "b:realmlp_te": 0.011039507507020605, "b:realmlp_te_cnt": -0.005583261278170029, "b:realmlp_te_s2": -0.006940507772879366, "b:realmlp_v1": 0.012630090172043857, "b:xgb_dig_d2": -0.02322669118450206, "b:xgb_raw_d3": 0.0014099646053971717, "b:xgb_te_d2": -0.0011439174998451335},
"cv_led": {"s:megayak_5f_lm": -0.0504101115868334, "s:masaya_gpt2": 0.01892536826920305, "s:lm7_5fold": 0.03268460975553137, "s:gpt2_seed2026": 0.00019156335507359648, "s:tabpfn_slimypunk_tp1": -0.012642649769990903, "s:sadam_tabpfn_100k_tp3": -0.03039229764870732, "s:yekenot_realmlp": -0.001156919333157887, "s:alex_glm_gen": -0.010049686607455182, "s:daoviet_gpdvae": 0.026524288367069906, "s:gbn_bag5_w512": 0.03197270577142789, "s:sameer_glm10f": 0.07539886111092914, "s:sadam_v6_gb_xgb": -0.03876423955389656, "s:sadam_v6_gb_lgb": 0.050619303134153054, "s:sadam_v4_cat": 0.024899735169078147, "s:gb_bag5": -0.012525566608465507, "s:megayak_shallow_tree": 0.08998587818302896, "wide": 0.05568936953093647, "b:ebm_all_bag": 0.01241314649800849, "b:ebm_dig": -0.033403049606166156, "b:ebm_dig_bag": 0.009782407957898523, "b:ebm_dig_r7": 0.0031988540661077067, "b:ebm_diglead_bag": -0.022134518874358366, "b:ebm_diglead_i25": 0.007513447059301521, "b:ebm_diglead_r11": 0.0004938077778375392, "b:ebm_diglead_r13": 0.014156212789682508, "b:ebm_i25_bag": -0.0147292301078183, "b:ebm_i25_bag4": 0.02385867724779454, "b:ebm_lead4_bag": 0.006688192218254173, "b:gb_bag10": -0.0015321268707586982, "b:gb_bag5": 0.0144427492994866, "b:gb_l3": -0.026744709811242964, "b:gb_ohk": -0.0250889728309019, "b:gbc_bag3": -0.0001499417896917808, "b:gbe_bag3": 0.007162876625302505, "b:gbn_bag5_w128": -0.008308739560439522, "b:gbn_bag5_w512": 0.01723221260596312, "b:gbr_bag3": -0.006241102802695989, "b:gbx_bag3": -0.006240817526956558, "b:gbx_bag5": -0.015449787795720182, "b:glm_dig": 0.0016094490785222377, "b:glm_dl": 0.001269407609764088, "b:glm_dl_bag": -0.019307803416807087, "b:glm_dl_s50": -0.016638174597513808, "b:glm_dl_s50_bag": -0.02308316908617957, "b:glm_ohk": -0.014920227472817757, "b:glm_ohk2": -0.010264756653736738, "b:glm_ohk2_bag": -0.022667189412379152, "b:glm_ohk_c01": -0.014560819249265932, "b:glm_ohkl": 0.012022636416038466, "b:glm_ohkl_bag": 0.010820057263254173, "b:glm_sb": -0.008585709230074623, "b:glm_te": 0.00014461870145916856, "b:lgb_dig_l4": 0.02775568242888243, "b:lgb_dig_l7": 0.01231196265590098, "b:lgb_dig_s100": -0.026963457242898014, "b:lgb_dig_s100_bag": -0.022771393714400665, "b:lgb_dig_s100_bag5": 0.01501939437317305, "b:lgb_dig_s100_l3": 0.020784028527905406, "b:lgb_dig_s100_l3_bag": 0.003401885062408753, "b:lgb_diglead_bag": -0.022936997298612535, "b:lgb_diglead_bag5": -0.026908092680123324, "b:lgb_diglead_l2": -0.0421340354076887, "b:lgb_diglead_l3": 0.04054503152925578, "b:lgb_diglead_l3_bag": -0.00040144349699201566, "b:lgb_diglead_l4": -0.00020162492642476296, "b:lgb_dk_l4": 0.0019528312307037553, "b:lgb_raw_l31": 0.0006356799380731614, "b:lgb_raw_l4": 0.006217819755726177, "b:lgb_raw_l4_cnt": -0.04126303032264386, "b:lgb_rawdig_l4": 0.01575670670651084, "b:lgb_te_l4": 0.018941213542591304, "b:lgb_te_l4_s100": 0.017060420694787578, "b:lgb_te_l4_s300": 0.0015753759554598585, "b:lgb_te_l4_s5": 0.0013086638671635535, "b:realmlp_allcat": -0.006125101859943052, "b:realmlp_cnt": 0.00544837093817107, "b:realmlp_dig": -0.014207768492579254, "b:realmlp_dig_s100": 0.005558545056911285, "b:realmlp_dig_s2": 0.012895054479795923, "b:realmlp_rawdig": -0.024402566794336375, "b:realmlp_te": 0.009895745291643016, "b:realmlp_te_cnt": -0.005084928725762969, "b:realmlp_te_s2": -0.006487850666131134, "b:realmlp_v1": 0.012200835041492056, "b:xgb_dig_d2": -0.03021531859263534, "b:xgb_raw_d3": 0.006410944962609165, "b:xgb_te_d2": -0.005833150956332071},
"step1": {"s:megayak_5f_lm": -0.003351112978983948, "s:masaya_gpt2": -0.004331262757969586, "s:lm7_5fold": 0.033366619003782044, "s:gpt2_seed2026": -0.007827509209426856, "s:tabpfn_slimypunk_tp1": -0.009713901949474838, "s:sadam_tabpfn_100k_tp3": -0.02258942534334505, "s:yekenot_realmlp": -0.004526829340289488, "s:alex_glm_gen": -0.00888266753715394, "s:daoviet_gpdvae": 0.029965005683580372, "s:gbn_bag5_w512": 0.034301067520051236, "s:sameer_glm10f": 0.032646833126727844, "s:sadam_v6_gb_xgb": -0.018135893519251264, "s:sadam_v6_gb_lgb": 0.016938816758909212, "s:sadam_v4_cat": 0.010122157702368371, "s:gb_bag5": -0.03439525156711189, "s:megayak_shallow_tree": 0.016529211843412543, "wide": 0.039253033212382056, "b:ebm_all_bag": 0.07523770624398651, "b:ebm_dig": -0.054123847865869386, "b:ebm_dig_bag": 0.042590023002053626, "b:ebm_dig_r7": -0.030431224464872567, "b:ebm_diglead_bag": -0.007475256474622878, "b:ebm_diglead_i25": 0.012130204546613841, "b:ebm_diglead_r11": -0.029545115889253553, "b:ebm_diglead_r13": -0.025314977657280278, "b:ebm_i25_bag": -0.034251702364764935, "b:ebm_i25_bag4": 0.0021819101691288286, "b:ebm_lead4_bag": 0.05402388421646351, "b:gb_bag10": 0.02127855874172903, "b:gb_bag5": 0.014346413654670603, "b:gb_l3": -0.013769934129875412, "b:gb_ohk": -0.01320976918577355, "b:gbc_bag3": 0.005252926825715231, "b:gbe_bag3": 0.0056608738989543635, "b:gbn_bag5_w128": -0.008066646084482498, "b:gbn_bag5_w512": 0.0027761714797112673, "b:gbr_bag3": -0.0015671193553017644, "b:gbx_bag3": 0.013676703301752498, "b:gbx_bag5": -0.024849339260214022, "b:glm_dig": 0.003075598457342358, "b:glm_dl": 0.010916567005561281, "b:glm_dl_bag": -0.02140273076626105, "b:glm_dl_s50": -0.0048270702443609145, "b:glm_dl_s50_bag": -0.028965516463380712, "b:glm_ohk": -0.009823478201748637, "b:glm_ohk2": -0.0318522897319079, "b:glm_ohk2_bag": 0.008008910874263413, "b:glm_ohk_c01": -0.001537026826110699, "b:glm_ohkl": -0.0007177995052908708, "b:glm_ohkl_bag": 0.02035140308004446, "b:glm_sb": -0.003981381639641186, "b:glm_te": 0.0003080122272580523, "b:lgb_dig_l4": 0.010603573387195964, "b:lgb_dig_l7": 0.004572758181607049, "b:lgb_dig_s100": -0.008401535545529524, "b:lgb_dig_s100_bag": -0.024937178041431557, "b:lgb_dig_s100_bag5": 0.009843322515543112, "b:lgb_dig_s100_l3": 0.005104687606392921, "b:lgb_dig_s100_l3_bag": 0.013836519222034662, "b:lgb_diglead_bag": -0.02825928718723698, "b:lgb_diglead_bag5": -0.00015918700995309258, "b:lgb_diglead_l2": -0.023477703221799207, "b:lgb_diglead_l3": 0.013687939454287347, "b:lgb_diglead_l3_bag": 0.01089390576912047, "b:lgb_diglead_l4": 0.0027356122650579223, "b:lgb_dk_l4": 0.0031170273617469727, "b:lgb_raw_l31": -0.0002778639224024611, "b:lgb_raw_l4": 0.005076659619331169, "b:lgb_raw_l4_cnt": -0.026000953337936953, "b:lgb_rawdig_l4": 0.008909983765388369, "b:lgb_te_l4": 0.014595251015497515, "b:lgb_te_l4_s100": 0.009202947491696535, "b:lgb_te_l4_s300": -0.0036164770792529123, "b:lgb_te_l4_s5": 0.004755908523758278, "b:realmlp_allcat": -0.009180037960367643, "b:realmlp_cnt": 0.002008152697517426, "b:realmlp_dig": -0.006650850028567239, "b:realmlp_dig_s100": 0.0011095904957665577, "b:realmlp_dig_s2": 0.011252248899267365, "b:realmlp_rawdig": -0.01135452534507476, "b:realmlp_te": 0.006400827919491363, "b:realmlp_te_cnt": -0.0023289922376294714, "b:realmlp_te_s2": 0.0004650667245475183, "b:realmlp_v1": 0.007421311984852427, "b:xgb_dig_d2": -0.014921295260094976, "b:xgb_raw_d3": 0.0029908808169654922, "b:xgb_te_d2": -0.004556637612790932},
"step2_half": {"s:megayak_5f_lm": 0.013560479235470527, "s:masaya_gpt2": -0.013415532419070966, "s:lm7_5fold": 0.016924808967985386, "s:gpt2_seed2026": -0.005846561584592116, "s:tabpfn_slimypunk_tp1": -0.00716915745504147, "s:sadam_tabpfn_100k_tp3": -0.012477860876310934, "s:yekenot_realmlp": -0.005847025911995225, "s:alex_glm_gen": -0.0035667420089754978, "s:daoviet_gpdvae": 0.010432227162500262, "s:gbn_bag5_w512": 0.03421288006649682, "s:sameer_glm10f": -0.01753859206217354, "s:sadam_v6_gb_xgb": -0.006008166073410063, "s:sadam_v6_gb_lgb": -0.011189574364595722, "s:sadam_v4_cat": -0.008215256938098263, "s:gb_bag5": -0.013234620290331862, "s:megayak_shallow_tree": -0.05329387696475422, "wide": 0.020055198517721345, "b:ebm_all_bag": 0.08032077305331266, "b:ebm_dig": -0.04821548653806729, "b:ebm_dig_bag": 0.0663608437089962, "b:ebm_dig_r7": -0.04763985976078816, "b:ebm_diglead_bag": 0.0012470873772310392, "b:ebm_diglead_i25": 0.020868319380952464, "b:ebm_diglead_r11": -0.018787199868178617, "b:ebm_diglead_r13": -0.031933987625123665, "b:ebm_i25_bag": -0.02852044811718722, "b:ebm_i25_bag4": -0.021637072121400334, "b:ebm_lead4_bag": 0.02950525709861906, "b:gb_bag10": 0.009815367106015185, "b:gb_bag5": 0.02334053299856081, "b:gb_l3": -0.006951219178063977, "b:gb_ohk": 0.0018332949866820256, "b:gbc_bag3": 0.0038870477010928325, "b:gbe_bag3": 0.0034865909557313356, "b:gbn_bag5_w128": -0.021110583050017305, "b:gbn_bag5_w512": 0.010671461592700661, "b:gbr_bag3": 0.011079141956471712, "b:gbx_bag3": 0.03516678094498248, "b:gbx_bag5": -0.022428290700120656, "b:glm_dig": 0.008039178685718558, "b:glm_dl": 0.010396749892529342, "b:glm_dl_bag": -0.014647055405793691, "b:glm_dl_s50": -0.0007624548737430984, "b:glm_dl_s50_bag": -0.03224603236752054, "b:glm_ohk": -0.0014554228489168505, "b:glm_ohk2": -0.021050754486035085, "b:glm_ohk2_bag": 0.014620275207319703, "b:glm_ohk_c01": 0.009039588532333274, "b:glm_ohkl": -0.015713624347275166, "b:glm_ohkl_bag": 0.03369452774708165, "b:glm_sb": 0.003223769038409277, "b:glm_te": -0.0003188723638362897, "b:lgb_dig_l4": -0.009902399919449884, "b:lgb_dig_l7": -0.006368773687195667, "b:lgb_dig_s100": 0.013405184345650493, "b:lgb_dig_s100_bag": -0.023787932186687104, "b:lgb_dig_s100_bag5": 0.006918520093627308, "b:lgb_dig_s100_l3": -0.008895789461048517, "b:lgb_dig_s100_l3_bag": 0.025983474774013818, "b:lgb_diglead_bag": -0.030799407825723078, "b:lgb_diglead_bag5": 0.026017077980131132, "b:lgb_diglead_l2": 0.0023658909444383616, "b:lgb_diglead_l3": -0.026516477795675772, "b:lgb_diglead_l3_bag": 0.024331011022829137, "b:lgb_diglead_l4": 0.012962785663650984, "b:lgb_dk_l4": 0.00705709120536385, "b:lgb_raw_l31": -0.0013077377856586425, "b:lgb_raw_l4": 0.003361630906731786, "b:lgb_raw_l4_cnt": -0.007073959269025274, "b:lgb_rawdig_l4": -0.0005792167740520289, "b:lgb_te_l4": 0.00777574722953822, "b:lgb_te_l4_s100": -0.005815187528929042, "b:lgb_te_l4_s300": -0.006128145913719751, "b:lgb_te_l4_s5": 0.0049920787148927635, "b:realmlp_allcat": -0.012708496532605128, "b:realmlp_cnt": -0.0007725462723897217, "b:realmlp_dig": 0.0024815610298947965, "b:realmlp_dig_s100": -0.0023806799843974443, "b:realmlp_dig_s2": 0.010535960646496067, "b:realmlp_rawdig": 0.008454413536396574, "b:realmlp_te": -0.0008520837876694293, "b:realmlp_te_cnt": 0.0019657911349665403, "b:realmlp_te_s2": 0.01204414552678721, "b:realmlp_v1": 0.0010588427054775687, "b:xgb_dig_d2": 4.381772846249486e-06, "b:xgb_raw_d3": 0.0017253470690219348, "b:xgb_te_d2": -0.006971556784199019},
"step2": {"s:megayak_5f_lm": 0.027120958470941054, "s:masaya_gpt2": -0.026831064838141933, "s:lm7_5fold": 0.03384961793597077, "s:gpt2_seed2026": -0.011693123169184232, "s:tabpfn_slimypunk_tp1": -0.01433831491008294, "s:sadam_tabpfn_100k_tp3": -0.024955721752621868, "s:yekenot_realmlp": -0.01169405182399045, "s:alex_glm_gen": -0.0071334840179509956, "s:daoviet_gpdvae": 0.020864454325000524, "s:gbn_bag5_w512": 0.06842576013299365, "s:sameer_glm10f": -0.03507718412434708, "s:sadam_v6_gb_xgb": -0.012016332146820127, "s:sadam_v6_gb_lgb": -0.022379148729191445, "s:sadam_v4_cat": -0.016430513876196526, "s:gb_bag5": -0.026469240580663723, "s:megayak_shallow_tree": -0.10658775392950844, "wide": 0.04011039703544269, "b:ebm_all_bag": 0.16064154610662532, "b:ebm_dig": -0.09643097307613457, "b:ebm_dig_bag": 0.1327216874179924, "b:ebm_dig_r7": -0.09527971952157632, "b:ebm_diglead_bag": 0.0024941747544620784, "b:ebm_diglead_i25": 0.04173663876190493, "b:ebm_diglead_r11": -0.037574399736357234, "b:ebm_diglead_r13": -0.06386797525024733, "b:ebm_i25_bag": -0.05704089623437444, "b:ebm_i25_bag4": -0.04327414424280067, "b:ebm_lead4_bag": 0.05901051419723812, "b:gb_bag10": 0.01963073421203037, "b:gb_bag5": 0.04668106599712162, "b:gb_l3": -0.013902438356127955, "b:gb_ohk": 0.0036665899733640513, "b:gbc_bag3": 0.007774095402185665, "b:gbe_bag3": 0.006973181911462671, "b:gbn_bag5_w128": -0.04222116610003461, "b:gbn_bag5_w512": 0.021342923185401323, "b:gbr_bag3": 0.022158283912943423, "b:gbx_bag3": 0.07033356188996497, "b:gbx_bag5": -0.04485658140024131, "b:glm_dig": 0.016078357371437115, "b:glm_dl": 0.020793499785058685, "b:glm_dl_bag": -0.029294110811587382, "b:glm_dl_s50": -0.0015249097474861967, "b:glm_dl_s50_bag": -0.06449206473504109, "b:glm_ohk": -0.002910845697833701, "b:glm_ohk2": -0.04210150897207017, "b:glm_ohk2_bag": 0.029240550414639407, "b:glm_ohk_c01": 0.018079177064666548, "b:glm_ohkl": -0.03142724869455033, "b:glm_ohkl_bag": 0.0673890554941633, "b:glm_sb": 0.006447538076818554, "b:glm_te": -0.0006377447276725794, "b:lgb_dig_l4": -0.019804799838899767, "b:lgb_dig_l7": -0.012737547374391334, "b:lgb_dig_s100": 0.026810368691300986, "b:lgb_dig_s100_bag": -0.04757586437337421, "b:lgb_dig_s100_bag5": 0.013837040187254616, "b:lgb_dig_s100_l3": -0.017791578922097033, "b:lgb_dig_s100_l3_bag": 0.051966949548027636, "b:lgb_diglead_bag": -0.061598815651446155, "b:lgb_diglead_bag5": 0.052034155960262264, "b:lgb_diglead_l2": 0.004731781888876723, "b:lgb_diglead_l3": -0.053032955591351544, "b:lgb_diglead_l3_bag": 0.048662022045658275, "b:lgb_diglead_l4": 0.025925571327301968, "b:lgb_dk_l4": 0.0141141824107277, "b:lgb_raw_l31": -0.002615475571317285, "b:lgb_raw_l4": 0.006723261813463572, "b:lgb_raw_l4_cnt": -0.014147918538050547, "b:lgb_rawdig_l4": -0.0011584335481040577, "b:lgb_te_l4": 0.01555149445907644, "b:lgb_te_l4_s100": -0.011630375057858085, "b:lgb_te_l4_s300": -0.012256291827439501, "b:lgb_te_l4_s5": 0.009984157429785527, "b:realmlp_allcat": -0.025416993065210256, "b:realmlp_cnt": -0.0015450925447794434, "b:realmlp_dig": 0.004963122059789593, "b:realmlp_dig_s100": -0.004761359968794889, "b:realmlp_dig_s2": 0.021071921292992134, "b:realmlp_rawdig": 0.016908827072793148, "b:realmlp_te": -0.0017041675753388585, "b:realmlp_te_cnt": 0.0039315822699330805, "b:realmlp_te_s2": 0.02408829105357442, "b:realmlp_v1": 0.0021176854109551373, "b:xgb_dig_d2": 8.763545692498972e-06, "b:xgb_raw_d3": 0.0034506941380438695, "b:xgb_te_d2": -0.013943113568398039},
"step3": {"s:megayak_5f_lm": 0.04291273505247678, "s:masaya_gpt2": -0.020199757317605413, "s:lm7_5fold": -0.01468694710136923, "s:gpt2_seed2026": 0.025130617752315624, "s:tabpfn_slimypunk_tp1": -0.004207018474268403, "s:sadam_tabpfn_100k_tp3": -0.007856503311289544, "s:yekenot_realmlp": -0.0035719921934940883, "s:alex_glm_gen": -0.00968299752240646, "s:daoviet_gpdvae": -0.0043104617980370685, "s:gbn_bag5_w512": 0.059432477779308095, "s:sameer_glm10f": -0.06276693941875201, "s:sadam_v6_gb_xgb": 0.022456148322158014, "s:sadam_v6_gb_lgb": -0.042356699610454136, "s:sadam_v4_cat": -0.039100958827360595, "s:gb_bag5": -0.012918528414324717, "s:megayak_shallow_tree": -0.14824366284669427, "wide": -0.03978368932406893, "b:ebm_all_bag": 0.1200398804268523, "b:ebm_dig": -0.041702477580910585, "b:ebm_dig_bag": 0.09583943650849604, "b:ebm_dig_r7": -0.08191239292938352, "b:ebm_diglead_bag": 0.01376425646113259, "b:ebm_diglead_i25": 0.04909794514164337, "b:ebm_diglead_r11": -0.011310427337500801, "b:ebm_diglead_r13": -0.057125015279515697, "b:ebm_i25_bag": -0.055536171086130987, "b:ebm_i25_bag4": -0.05374364216150403, "b:ebm_lead4_bag": 0.02357861276219015, "b:gb_bag10": -0.00154617797821281, "b:gb_bag5": 0.05069096767529185, "b:gb_l3": 0.004029341040004309, "b:gb_ohk": 0.01068889956540223, "b:gbc_bag3": -0.0005705614977412649, "b:gbe_bag3": 0.006611024205409296, "b:gbn_bag5_w128": -0.05704424635393862, "b:gbn_bag5_w512": 0.018012229412251064, "b:gbr_bag3": 0.03408846711277557, "b:gbx_bag3": 0.08559259050915491, "b:gbx_bag5": -0.039653789628059245, "b:glm_dig": 0.02116955140650671, "b:glm_dl": 0.004080087233724273, "b:glm_dl_bag": -0.027410071095152855, "b:glm_dl_s50": -0.004560532985992997, "b:glm_dl_s50_bag": -0.04112787498159549, "b:glm_ohk": -0.005032986969082324, "b:glm_ohk2": -0.029147458944055227, "b:glm_ohk2_bag": 0.02415972281018389, "b:glm_ohk_c01": 0.012367133647285219, "b:glm_ohkl": 7.992068950317359e-05, "b:glm_ohkl_bag": 0.051688750332465824, "b:glm_sb": 0.014598944763857322, "b:glm_te": -0.0012856961951914344, "b:lgb_dig_l4": -0.044067513945396465, "b:lgb_dig_l7": -0.023937328022709262, "b:lgb_dig_s100": 0.05015416783062814, "b:lgb_dig_s100_bag": -0.0490099953747574, "b:lgb_dig_s100_bag5": 0.013292507325267445, "b:lgb_dig_s100_l3": -0.035535165439837854, "b:lgb_dig_s100_l3_bag": 0.06438780421173648, "b:lgb_diglead_bag": -0.06663793554784707, "b:lgb_diglead_bag5": 0.08463917295571141, "b:lgb_diglead_l2": 0.03429174384535712, "b:lgb_diglead_l3": -0.0935496417164133, "b:lgb_diglead_l3_bag": 0.05362307552217068, "b:lgb_diglead_l4": 0.0362732151219765, "b:lgb_dk_l4": 0.01552350185504524, "b:lgb_raw_l31": -0.00014661012254933467, "b:lgb_raw_l4": 0.0055072120668919235, "b:lgb_raw_l4_cnt": 0.007272332997137415, "b:lgb_rawdig_l4": -0.010187873970101267, "b:lgb_te_l4": 0.012398979732044752, "b:lgb_te_l4_s100": -0.019261819401553137, "b:lgb_te_l4_s300": -0.01860262372588006, "b:lgb_te_l4_s5": -0.0009787320202008865, "b:realmlp_allcat": -0.023775521024247927, "b:realmlp_cnt": -0.004851163257622534, "b:realmlp_dig": 0.014532461400033083, "b:realmlp_dig_s100": -0.012877357633809388, "b:realmlp_dig_s2": 0.019550644054602478, "b:realmlp_rawdig": 0.03289924143580761, "b:realmlp_te": -0.00795501732687936, "b:realmlp_te_cnt": 0.007044519774050301, "b:realmlp_te_s2": 0.030264994780156384, "b:realmlp_v1": -0.0023391743983786485, "b:xgb_dig_d2": 0.01606411465109771, "b:xgb_raw_d3": -0.0012450782349890336, "b:xgb_te_d2": -0.010690978580007995},
"step4": {"s:megayak_5f_lm": -0.0034888369366524582, "s:masaya_gpt2": -0.015284738285236195, "s:lm7_5fold": -0.03949878826858231, "s:gpt2_seed2026": 0.06946230892735698, "s:tabpfn_slimypunk_tp1": 0.004363023306568827, "s:sadam_tabpfn_100k_tp3": -0.013135678637731263, "s:yekenot_realmlp": 0.011539353358612734, "s:alex_glm_gen": -0.02889340863736857, "s:daoviet_gpdvae": -0.003158411394969963, "s:gbn_bag5_w512": 0.07429864312399857, "s:sameer_glm10f": 0.01083020494444985, "s:sadam_v6_gb_xgb": 0.06192083641586277, "s:sadam_v6_gb_lgb": -0.014245809297054237, "s:sadam_v4_cat": -0.04249350596858786, "s:gb_bag5": -0.018690888456457076, "s:megayak_shallow_tree": -0.035485976486572604, "wide": -0.03894421088390168, "b:ebm_all_bag": 0.03677181337958047, "b:ebm_dig": 0.02417920459799613, "b:ebm_dig_bag": 0.018801516920721822, "b:ebm_dig_r7": -0.04218759710577042, "b:ebm_diglead_bag": 0.010635010029495755, "b:ebm_diglead_i25": 0.04957738213731999, "b:ebm_diglead_r11": 0.026793560344778228, "b:ebm_diglead_r13": -0.018667093320712, "b:ebm_i25_bag": -0.049075610409961654, "b:ebm_i25_bag4": -0.03384809312970651, "b:ebm_lead4_bag": -0.015195704080086938, "b:gb_bag10": -0.03510906450687058, "b:gb_bag5": 0.045443272119950036, "b:gb_l3": 0.011155345916963714, "b:gb_ohk": -0.018980301466611775, "b:gbc_bag3": -0.012156266174697165, "b:gbe_bag3": 0.01731944865242502, "b:gbn_bag5_w128": -0.06436335154682796, "b:gbn_bag5_w512": 0.019785734139272378, "b:gbr_bag3": 0.029508964640899294, "b:gbx_bag3": 0.07462465072432652, "b:gbx_bag5": -0.03848657908060729, "b:glm_dig": 0.02412632595931039, "b:glm_dl": -0.03997507254154347, "b:glm_dl_bag": -0.041032343432782496, "b:glm_dl_s50": -0.009150197988898694, "b:glm_dl_s50_bag": -0.025061916274043687, "b:glm_ohk": -0.03897688288809032, "b:glm_ohk2": -0.023966298606199914, "b:glm_ohk2_bag": -0.022134715832265495, "b:glm_ohk_c01": -0.021532163362983796, "b:glm_ohkl": 0.06424804151643952, "b:glm_ohkl_bag": 0.04655505223495522, "b:glm_sb": 0.013799322854010239, "b:glm_te": -0.00013493658915205456, "b:lgb_dig_l4": -0.045692432389347906, "b:lgb_dig_l7": -0.02324972627301555, "b:lgb_dig_s100": 0.04486202594296171, "b:lgb_dig_s100_bag": -0.06543446351610092, "b:lgb_dig_s100_bag5": 0.017326167041775087, "b:lgb_dig_s100_l3": -0.039522775606475706, "b:lgb_dig_s100_l3_bag": 0.07698351772220281, "b:lgb_diglead_bag": -0.0918785984210468, "b:lgb_diglead_bag5": 0.10136280283757386, "b:lgb_diglead_l2": 0.026979966112458703, "b:lgb_diglead_l3": -0.08544471791540253, "b:lgb_diglead_l3_bag": 0.03991895600315462, "b:lgb_diglead_l4": 0.039131568140953664, "b:lgb_dk_l4": 0.01398481072426439, "b:lgb_raw_l31": 0.008530204922566535, "b:lgb_raw_l4": 0.009975270088348644, "b:lgb_raw_l4_cnt": -0.011305223356159364, "b:lgb_rawdig_l4": -0.004428883454854229, "b:lgb_te_l4": 0.03588414217788614, "b:lgb_te_l4_s100": 0.004402819123102933, "b:lgb_te_l4_s300": -0.033946661892970076, "b:lgb_te_l4_s5": -0.020379548644496457, "b:realmlp_allcat": -0.021847945261691686, "b:realmlp_cnt": -0.005115497538302371, "b:realmlp_dig": 0.009231675087523289, "b:realmlp_dig_s100": -0.021105687058130693, "b:realmlp_dig_s2": 0.026386424981860487, "b:realmlp_rawdig": 0.016088000296832713, "b:realmlp_te": -0.0009418848153852933, "b:realmlp_te_cnt": 0.002193161082475847, "b:realmlp_te_s2": 0.020374264481207105, "b:realmlp_v1": 0.01134837014926146, "b:xgb_dig_d2": 0.008104427329574886, "b:xgb_raw_d3": -0.0071573342074869525, "b:xgb_te_d2": -0.0021432682839698687},
"step5": {"s:megayak_5f_lm": 0.06897650738977462, "s:masaya_gpt2": -0.009015859365062587, "s:lm7_5fold": 0.0009215367354582742, "s:gpt2_seed2026": -0.010251325714452565, "s:tabpfn_slimypunk_tp1": 0.0141561368318242, "s:sadam_tabpfn_100k_tp3": 0.003507144176651289, "s:yekenot_realmlp": 0.000554288971888248, "s:alex_glm_gen": -0.01799965802469446, "s:daoviet_gpdvae": -0.00757453253363324, "s:gbn_bag5_w512": 0.11668761153993083, "s:sameer_glm10f": -0.039776089842193205, "s:sadam_v6_gb_xgb": 0.08655821449241188, "s:sadam_v6_gb_lgb": -0.06304259940783537, "s:sadam_v4_cat": -0.033421792690255456, "s:gb_bag5": -0.04673261552510822, "s:megayak_shallow_tree": -0.1253885827427653, "wide": -0.013401948752305786, "b:ebm_all_bag": 0.16535217569905436, "b:ebm_dig": -0.06466504774415102, "b:ebm_dig_bag": 0.10939458360630332, "b:ebm_dig_r7": -0.10904054532975026, "b:ebm_diglead_bag": 0.15358016925677073, "b:ebm_diglead_i25": 0.03882931804560387, "b:ebm_diglead_r11": -0.10797963213618902, "b:ebm_diglead_r13": -0.141475691530386, "b:ebm_i25_bag": -0.03330261397321142, "b:ebm_i25_bag4": -0.056408685897294995, "b:ebm_lead4_bag": 0.0464173963220906, "b:gb_bag10": 0.03194951889707664, "b:gb_bag5": 0.02047301464717305, "b:gb_l3": 0.05570343463674305, "b:gb_ohk": -0.013256060365589234, "b:gbc_bag3": 0.00530342071496869, "b:gbe_bag3": 0.00273180802397477, "b:gbn_bag5_w128": -0.023383619716230583, "b:gbn_bag5_w512": -0.07096680647838312, "b:gbr_bag3": 0.009083751637504819, "b:gbx_bag3": 0.1020836624830321, "b:gbx_bag5": -0.10372444459039258, "b:glm_dig": 0.013474305968518285, "b:glm_dl": -0.032059274008832454, "b:glm_dl_bag": -0.021473603925046127, "b:glm_dl_s50": 0.020583262666812933, "b:glm_dl_s50_bag": -0.0039183901110275395, "b:glm_ohk": -0.03765485928672982, "b:glm_ohk2": -0.08255453451274214, "b:glm_ohk2_bag": 0.04653641203627543, "b:glm_ohk_c01": 0.013055097738295897, "b:glm_ohkl": 0.06292326173214656, "b:glm_ohkl_bag": 0.021615313062059504, "b:glm_sb": 0.009721586222763782, "b:glm_te": -0.006725737063415938, "b:lgb_dig_l4": -0.055837477082973985, "b:lgb_dig_l7": -0.016350818439952525, "b:lgb_dig_s100": 0.058211729634193396, "b:lgb_dig_s100_bag": -0.06111204912040655, "b:lgb_dig_s100_bag5": -0.0073488267115739215, "b:lgb_dig_s100_l3": -0.06170331263545208, "b:lgb_dig_s100_l3_bag": 0.07503784806078012, "b:lgb_diglead_bag": -0.1028286243985103, "b:lgb_diglead_bag5": 0.13560974793062627, "b:lgb_diglead_l2": 0.03318288998272117, "b:lgb_diglead_l3": -0.07449187136326936, "b:lgb_diglead_l3_bag": 0.03916999201821208, "b:lgb_diglead_l4": 0.02465532784154723, "b:lgb_dk_l4": 0.007398935534430129, "b:lgb_raw_l31": 0.0065574200392839775, "b:lgb_raw_l4": 0.0029355975964231408, "b:lgb_raw_l4_cnt": 0.013767449345681268, "b:lgb_rawdig_l4": -0.004514260114381885, "b:lgb_te_l4": 0.01348779111629848, "b:lgb_te_l4_s100": 0.01580164481063912, "b:lgb_te_l4_s300": -0.036475794446526356, "b:lgb_te_l4_s5": 0.0010410112240718519, "b:realmlp_allcat": -0.011665638181812679, "b:realmlp_cnt": -0.013742954748969567, "b:realmlp_dig": 0.008894012891697738, "b:realmlp_dig_s100": -0.02251635410884371, "b:realmlp_dig_s2": 0.00834843745840769, "b:realmlp_rawdig": 0.01743081519880163, "b:realmlp_te": 0.0034764493021211245, "b:realmlp_te_cnt": 0.0026581217023631744, "b:realmlp_te_s2": 0.015409114581696826, "b:realmlp_v1": 0.005522739951764543, "b:xgb_dig_d2": 0.028929272398427484, "b:xgb_raw_d3": -0.016849732900939916, "b:xgb_te_d2": 0.005379140652384981},
"step6": {"s:megayak_5f_lm": -0.11167117542101894, "s:masaya_gpt2": 0.008580731570136326, "s:lm7_5fold": -0.0009400463836016774, "s:gpt2_seed2026": 0.027885869579209244, "s:tabpfn_slimypunk_tp1": 0.0016924449935887891, "s:sadam_tabpfn_100k_tp3": 0.01432901271297176, "s:yekenot_realmlp": 0.022390415632839642, "s:alex_glm_gen": -0.0005758371133733123, "s:daoviet_gpdvae": 0.01311917773336681, "s:gbn_bag5_w512": -0.009545946811432997, "s:sameer_glm10f": 0.05630282825450275, "s:sadam_v6_gb_xgb": -0.08321586634169159, "s:sadam_v6_gb_lgb": 0.10171308794729168, "s:sadam_v4_cat": 0.03041043931485113, "s:gb_bag5": -0.013425667860656836, "s:megayak_shallow_tree": 0.18113423530486364, "wide": 0.03224417977782197, "b:ebm_all_bag": -0.03928670360117938, "b:ebm_dig": 0.015432582908453892, "b:ebm_dig_bag": -0.019730929338460986, "b:ebm_dig_r7": 0.021228118735024122, "b:ebm_diglead_bag": -0.019405585095929167, "b:ebm_diglead_i25": -0.016076869397068764, "b:ebm_diglead_r11": 0.019304661106694525, "b:ebm_diglead_r13": 0.025249635776052333, "b:ebm_i25_bag": -0.016462397253955638, "b:ebm_i25_bag4": 0.04030328340809125, "b:ebm_lead4_bag": -0.046473301351245, "b:gb_bag10": -0.004465127493976626, "b:gb_bag5": 0.005295157073829479, "b:gb_l3": -0.07872430096932825, "b:gb_ohk": 0.007827567154987038, "b:gbc_bag3": -0.03280747819608797, "b:gbe_bag3": 0.01175576219083039, "b:gbn_bag5_w128": 0.018717264487127714, "b:gbn_bag5_w512": -0.0011920109259677498, "b:gbr_bag3": 0.004896649909764378, "b:gbx_bag3": -0.011667422511690605, "b:gbx_bag5": 0.015530360868756142, "b:glm_dig": 0.004036061702293574, "b:glm_dl": 0.04219485215779119, "b:glm_dl_bag": 0.010864644604441424, "b:glm_dl_s50": -0.008276003887968196, "b:glm_dl_s50_bag": -0.05826467282773935, "b:glm_ohk": 0.043882937000895586, "b:glm_ohk2": 0.02848919574777289, "b:glm_ohk2_bag": -0.009287167708046554, "b:glm_ohk_c01": -0.03407033309595358, "b:glm_ohkl": -0.013764349514068144, "b:glm_ohkl_bag": -0.03603962101306747, "b:glm_sb": -0.0023173698639354762, "b:glm_te": -0.0009163310748722242, "b:lgb_dig_l4": 0.06490818309252397, "b:lgb_dig_l7": 0.005332908943789934, "b:lgb_dig_s100": -0.039099926221213735, "b:lgb_dig_s100_bag": 0.021086002265371388, "b:lgb_dig_s100_bag5": 0.018067651487240086, "b:lgb_dig_s100_l3": 0.050308694696048394, "b:lgb_dig_s100_l3_bag": -0.021781737915879996, "b:lgb_diglead_bag": 0.025863505660522864, "b:lgb_diglead_bag5": -0.09141971792291925, "b:lgb_diglead_l2": -0.03604788078810194, "b:lgb_diglead_l3": 0.039415925471774575, "b:lgb_diglead_l3_bag": -0.023375313510179387, "b:lgb_diglead_l4": 0.005674349495556749, "b:lgb_dk_l4": 0.0017674212665991548, "b:lgb_raw_l31": -0.004471455199590837, "b:lgb_raw_l4": -0.0036951771362210197, "b:lgb_raw_l4_cnt": -0.053748077290576096, "b:lgb_rawdig_l4": 0.005242284896971225, "b:lgb_te_l4": -0.021624575044207693, "b:lgb_te_l4_s100": -0.01718882590212164, "b:lgb_te_l4_s300": 0.03017029260151744, "b:lgb_te_l4_s5": -0.02436183442707204, "b:realmlp_allcat": 0.013600667820710632, "b:realmlp_cnt": 0.019958712814710327, "b:realmlp_dig": -0.0076563752581303365, "b:realmlp_dig_s100": 0.023659717425888894, "b:realmlp_dig_s2": -0.0029996770123733116, "b:realmlp_rawdig": -0.013394120120286176, "b:realmlp_te": -0.013346668023044723, "b:realmlp_te_cnt": -0.0017512931976057325, "b:realmlp_te_s2": -0.0175153416436808, "b:realmlp_v1": -0.0004147861382334795, "b:xgb_dig_d2": -0.04178865951042272, "b:xgb_raw_d3": 0.004541591163347932, "b:xgb_te_d2": -0.0006877082372638679},
"step7": {"s:megayak_5f_lm": -0.01532889506612532, "s:masaya_gpt2": -0.013261391079182689, "s:lm7_5fold": 0.004874633623866584, "s:gpt2_seed2026": -0.011048328252712398, "s:tabpfn_slimypunk_tp1": 0.019479195136036475, "s:sadam_tabpfn_100k_tp3": 0.03379897098415831, "s:yekenot_realmlp": 0.029448884712266525, "s:alex_glm_gen": -0.04076182290815198, "s:daoviet_gpdvae": 0.03885285756599925, "s:gbn_bag5_w512": 0.016705008805907157, "s:sameer_glm10f": -0.033671768825366336, "s:sadam_v6_gb_xgb": 0.005793147308091748, "s:sadam_v6_gb_lgb": 0.035878360062331686, "s:sadam_v4_cat": -0.014177348042369899, "s:gb_bag5": -0.002983076133072762, "s:megayak_shallow_tree": 0.0429774041829996, "wide": -0.0012445682470603396, "b:ebm_all_bag": 0.011416914599689673, "b:ebm_dig": 0.02294984166507186, "b:ebm_dig_bag": 0.028877548053109183, "b:ebm_dig_r7": -0.04276285768277372, "b:ebm_diglead_bag": -0.008037767147017849, "b:ebm_diglead_i25": 0.019575714007213182, "b:ebm_diglead_r11": 0.005822450818036772, "b:ebm_diglead_r13": -0.03646698657915979, "b:ebm_i25_bag": -0.045582127670408, "b:ebm_i25_bag4": -0.023571440063628404, "b:ebm_lead4_bag": 0.0024724934563454997, "b:gb_bag10": 0.010602709995019643, "b:gb_bag5": -0.00020948904926665563, "b:gb_l3": -0.005008883794821487, "b:gb_ohk": 0.012462403114245438, "b:gbc_bag3": -0.04491596636131862, "b:gbe_bag3": 0.021281358413983074, "b:gbn_bag5_w128": 0.03773384363936814, "b:gbn_bag5_w512": -0.042487368245716466, "b:gbr_bag3": 0.02379136220071678, "b:gbx_bag3": 0.0801743069671423, "b:gbx_bag5": -0.06676185994403352, "b:glm_dig": 0.018579401866817363, "b:glm_dl": 0.047057489740079596, "b:glm_dl_bag": 0.012129654721577742, "b:glm_dl_s50": -0.016842606244825434, "b:glm_dl_s50_bag": -0.046154572849136107, "b:glm_ohk": 0.0279298113811889, "b:glm_ohk2": -0.005464711505301273, "b:glm_ohk2_bag": -0.00444217654093287, "b:glm_ohk_c01": -0.02521683001276968, "b:glm_ohkl": 0.00998815286349657, "b:glm_ohkl_bag": 0.011188677652772922, "b:glm_sb": 0.011534339980283794, "b:glm_te": -0.01941353864783445, "b:lgb_dig_l4": 0.012792179508792752, "b:lgb_dig_l7": -0.016536600806802363, "b:lgb_dig_s100": 0.025988292377181515, "b:lgb_dig_s100_bag": -0.02786056607370909, "b:lgb_dig_s100_bag5": 0.004711599818220775, "b:lgb_dig_s100_l3": -0.018316961517069283, "b:lgb_dig_s100_l3_bag": 0.05854090458349093, "b:lgb_diglead_bag": -0.058955263331882175, "b:lgb_diglead_bag5": 0.042922577252749214, "b:lgb_diglead_l2": -0.0006251518955284371, "b:lgb_diglead_l3": -0.04723462060156513, "b:lgb_diglead_l3_bag": 0.005509594253736563, "b:lgb_diglead_l4": 0.025174373862716728, "b:lgb_dk_l4": 0.015083685645311297, "b:lgb_raw_l31": 0.008250997152138546, "b:lgb_raw_l4": -0.02098204469918052, "b:lgb_raw_l4_cnt": -0.04670494103234027, "b:lgb_rawdig_l4": 0.0004929563863006859, "b:lgb_te_l4": -0.02137609804945939, "b:lgb_te_l4_s100": -0.002944265144476053, "b:lgb_te_l4_s300": -0.012556895046753664, "b:lgb_te_l4_s5": -0.030877460829031114, "b:realmlp_allcat": 0.01652923534649207, "b:realmlp_cnt": 0.005679425026097773, "b:realmlp_dig": 0.00040613096397866576, "b:realmlp_dig_s100": 0.0021495226045041803, "b:realmlp_dig_s2": -0.006397105937466286, "b:realmlp_rawdig": 0.005301293537676727, "b:realmlp_te": -0.018820769127796483, "b:realmlp_te_cnt": 0.007281638359193587, "b:realmlp_te_s2": -0.013005975189400712, "b:realmlp_v1": 0.007037322452335817, "b:xgb_dig_d2": 0.015738726611111017, "b:xgb_raw_d3": -0.031273627583305666, "b:xgb_te_d2": 0.01834167087631238},
"step8": {"s:megayak_5f_lm": -0.04109209779006236, "s:masaya_gpt2": -0.0004072271721775545, "s:lm7_5fold": 0.01677508950219521, "s:gpt2_seed2026": -0.00454561149714024, "s:tabpfn_slimypunk_tp1": -0.014977759241976045, "s:sadam_tabpfn_100k_tp3": 0.029716037850354433, "s:yekenot_realmlp": 0.016541903564940435, "s:alex_glm_gen": -0.09040904532301039, "s:daoviet_gpdvae": 0.017065440386248958, "s:gbn_bag5_w512": -0.013515567121968218, "s:sameer_glm10f": 0.12354214111676283, "s:sadam_v6_gb_xgb": 0.016585823564472444, "s:sadam_v6_gb_lgb": 0.05884571943536787, "s:sadam_v4_cat": 0.013683348925251805, "s:gb_bag5": -0.01792064159069167, "s:megayak_shallow_tree": 0.21636817764873012, "wide": 0.04065066663469026, "b:ebm_all_bag": 0.03974631939603844, "b:ebm_dig": -0.052419036975476695, "b:ebm_dig_bag": 0.027702467279231528, "b:ebm_dig_r7": -0.03686164649044866, "b:ebm_diglead_bag": -0.01622252534639919, "b:ebm_diglead_i25": -0.06834767018947224, "b:ebm_diglead_r11": 0.005741738589489668, "b:ebm_diglead_r13": 0.005911002719386395, "b:ebm_i25_bag": 0.04163991410040231, "b:ebm_i25_bag4": 0.010434245831339665, "b:ebm_lead4_bag": 0.018857617170889518, "b:gb_bag10": 0.021562865453753004, "b:gb_bag5": 0.02683653841535095, "b:gb_l3": -0.050722743000320064, "b:gb_ohk": 0.03306453132423512, "b:gbc_bag3": 0.050739912388960424, "b:gbe_bag3": 0.03832646518901568, "b:gbn_bag5_w128": -0.038327010961284144, "b:gbn_bag5_w512": 0.0018772633332485075, "b:gbr_bag3": 0.004252704146358474, "b:gbx_bag3": -0.03681435310534449, "b:gbx_bag5": -0.030584931926101715, "b:glm_dig": -0.025665264883203982, "b:glm_dl": -0.07457528803838122, "b:glm_dl_bag": -0.0462662355628784, "b:glm_dl_s50": 0.004130742810772969, "b:glm_dl_s50_bag": 0.04214621067076584, "b:glm_ohk": -0.13968000080957665, "b:glm_ohk2": 0.02725169321419692, "b:glm_ohk2_bag": 0.001046215771534468, "b:glm_ohk_c01": 0.1055669188810561, "b:glm_ohkl": -0.004646537483636044, "b:glm_ohkl_bag": 0.0017336826401156157, "b:glm_sb": -0.049122170851743245, "b:glm_te": -0.009640170627664509, "b:lgb_dig_l4": 0.03683047040700145, "b:lgb_dig_l7": -0.017971858116997832, "b:lgb_dig_s100": -0.0643911468203676, "b:lgb_dig_s100_bag": 0.013075779167918447, "b:lgb_dig_s100_bag5": 0.03724769072761586, "b:lgb_dig_s100_l3": 0.0018834692166121892, "b:lgb_dig_s100_l3_bag": -0.022582120383903594, "b:lgb_diglead_bag": 0.01595182982787076, "b:lgb_diglead_bag5": -0.03933392334154019, "b:lgb_diglead_l2": 0.0016450423149382924, "b:lgb_diglead_l3": 0.02440017314450028, "b:lgb_diglead_l3_bag": 0.010515317984925884, "b:lgb_diglead_l4": -0.017067698861575786, "b:lgb_dk_l4": -0.020038891246854606, "b:lgb_raw_l31": -0.037263189016030075, "b:lgb_raw_l4": 0.04273134130063517, "b:lgb_raw_l4_cnt": -0.04235618250857367, "b:lgb_rawdig_l4": 0.010748092961550344, "b:lgb_te_l4": 0.01301396286932004, "b:lgb_te_l4_s100": 0.007368004424707277, "b:lgb_te_l4_s300": 0.0153424938864005, "b:lgb_te_l4_s5": -0.014841100568703188, "b:realmlp_allcat": -0.013921397115458064, "b:realmlp_cnt": 0.014027120063129516, "b:realmlp_dig": -0.011479770558927002, "b:realmlp_dig_s100": 0.01513352400874948, "b:realmlp_dig_s2": -0.00018398578427414076, "b:realmlp_rawdig": -0.008906111575345682, "b:realmlp_te": 0.01878586990057515, "b:realmlp_te_cnt": -0.03409469975108967, "b:realmlp_te_s2": -0.008673831653114425, "b:realmlp_v1": -0.047452376637052006, "b:xgb_dig_d2": -0.022613738568250972, "b:xgb_raw_d3": 0.0287270533363839, "b:xgb_te_d2": 0.04452605695928168},
"step9": {"s:megayak_5f_lm": -0.03801779690054838, "s:masaya_gpt2": -0.012398757159749176, "s:lm7_5fold": 0.012866005941488168, "s:gpt2_seed2026": -0.007914261325950415, "s:tabpfn_slimypunk_tp1": -0.009658713570926979, "s:sadam_tabpfn_100k_tp3": 0.026429646954609157, "s:yekenot_realmlp": 0.0143107148800771, "s:alex_glm_gen": -0.0715849151594359, "s:daoviet_gpdvae": 0.011752862259258123, "s:gbn_bag5_w512": -0.02901320919496776, "s:sameer_glm10f": 0.09051095899369324, "s:sadam_v6_gb_xgb": 0.02943596670681914, "s:sadam_v6_gb_lgb": 0.030646617875032204, "s:sadam_v4_cat": -0.0024610114809747205, "s:gb_bag5": -0.08263252827224613, "s:megayak_shallow_tree": 0.21078246405321463, "wide": 0.019734003811161787, "b:ebm_all_bag": 0.16796352709326515, "b:ebm_dig": -0.12561550731983628, "b:ebm_dig_bag": 0.13934782775066704, "b:ebm_dig_r7": -0.1261142494833071, "b:ebm_diglead_bag": -0.07217332404648022, "b:ebm_diglead_i25": -0.07719633515896142, "b:ebm_diglead_r11": 0.007829566640033887, "b:ebm_diglead_r13": -0.014543994714038175, "b:ebm_i25_bag": 0.06087187703969249, "b:ebm_i25_bag4": -0.012792765815688945, "b:ebm_lead4_bag": 0.029710739596442128, "b:gb_bag10": 0.06304085759086593, "b:gb_bag5": 0.03660805652300675, "b:gb_l3": -0.03924484171864036, "b:gb_ohk": 0.024595736967937952, "b:gbc_bag3": 0.04038650012277033, "b:gbe_bag3": 0.033683063706469854, "b:gbn_bag5_w128": -0.030247795996085836, "b:gbn_bag5_w512": 0.02667942321382021, "b:gbr_bag3": 0.004561576741806142, "b:gbx_bag3": -0.024616706195567748, "b:gbx_bag5": -0.037558186200247895, "b:glm_dig": -0.012928437157816136, "b:glm_dl": -0.07431791103098741, "b:glm_dl_bag": -0.057231766165417995, "b:glm_dl_s50": -0.04751648846694765, "b:glm_dl_s50_bag": 0.10792169831841972, "b:glm_ohk": -0.1297207335069542, "b:glm_ohk2": 0.0525386469301767, "b:glm_ohk2_bag": -0.045097980488564005, "b:glm_ohk_c01": 0.11267317096515399, "b:glm_ohkl": -0.007590510066256837, "b:glm_ohkl_bag": 0.022810793574183617, "b:glm_sb": -0.036771821946326826, "b:glm_te": -0.010690748191414984, "b:lgb_dig_l4": 0.014149470866641665, "b:lgb_dig_l7": -0.0194791521819369, "b:lgb_dig_s100": -0.039219952031966265, "b:lgb_dig_s100_bag": 0.006086568037476102, "b:lgb_dig_s100_bag5": 0.030566772595538187, "b:lgb_dig_s100_l3": -0.005276509720214608, "b:lgb_dig_s100_l3_bag": -0.018408094807796472, "b:lgb_diglead_bag": 0.00852375519204105, "b:lgb_diglead_bag5": -0.016594574340684013, "b:lgb_diglead_l2": 0.013905155560229305, "b:lgb_diglead_l3": -0.0026491185214467133, "b:lgb_diglead_l3_bag": 0.019525802249101207, "b:lgb_diglead_l4": -0.011078506183525332, "b:lgb_dk_l4": -0.013373308939492679, "b:lgb_raw_l31": -0.02985793130600756, "b:lgb_raw_l4": 0.042057121249940016, "b:lgb_raw_l4_cnt": -0.03863528102995344, "b:lgb_rawdig_l4": 0.002901495114917194, "b:lgb_te_l4": 0.011055834032033736, "b:lgb_te_l4_s100": 0.0045954256822040036, "b:lgb_te_l4_s300": 0.005862107796361839, "b:lgb_te_l4_s5": -0.019187118532553525, "b:realmlp_allcat": -0.013475445854302744, "b:realmlp_cnt": 0.011010790412391587, "b:realmlp_dig": -0.005527529242210941, "b:realmlp_dig_s100": 0.006973439710724587, "b:realmlp_dig_s2": 0.0009578781020233357, "b:realmlp_rawdig": -0.0020842985473678993, "b:realmlp_te": 0.013000894479408106, "b:realmlp_te_cnt": -0.02699543668320104, "b:realmlp_te_s2": -0.0006761661897169103, "b:realmlp_v1": -0.03914446955537195, "b:xgb_dig_d2": -0.008929031702745673, "b:xgb_raw_d3": 0.015426288915169643, "b:xgb_te_d2": 0.042383088502402445}
}'''
# Version 8: step 10, the same 87 members plus alexchilton's stack file (x:Sold)
STEP10_JSON = '''{"s:megayak_5f_lm": -0.10004244088252182, "s:masaya_gpt2": -0.11872161427871783, "s:lm7_5fold": 0.01041157517420295, "s:gpt2_seed2026": -0.025258136738351162, "s:tabpfn_slimypunk_tp1": 0.010718722418205068, "s:sadam_tabpfn_100k_tp3": -0.027449859910952248, "s:yekenot_realmlp": -0.03138947947898212, "s:alex_glm_gen": 0.0767519065769284, "s:daoviet_gpdvae": 0.0339732463108965, "s:gbn_bag5_w512": 0.00439156803806506, "s:sameer_glm10f": -0.08691292137840573, "s:sadam_v6_gb_xgb": -0.04491626705819507, "s:sadam_v6_gb_lgb": -0.05474587029173319, "s:sadam_v4_cat": -0.017579583177393606, "s:gb_bag5": -0.03430190609640302, "s:megayak_shallow_tree": 0.10521453750800527, "wide": 0.004463224444272593, "b:ebm_all_bag": 0.12258218939929189, "b:ebm_dig": -0.05408025687736924, "b:ebm_dig_bag": 0.10466111674683422, "b:ebm_dig_r7": -0.06846322073804074, "b:ebm_diglead_bag": -0.06051834321992415, "b:ebm_diglead_i25": -0.057538182747044024, "b:ebm_diglead_r11": -0.008924606656955148, "b:ebm_diglead_r13": -0.06719918383979652, "b:ebm_i25_bag": 0.040956831155859075, "b:ebm_i25_bag4": -0.047232569464555306, "b:ebm_lead4_bag": 0.08062588366702272, "b:gb_bag10": 0.002768109857215111, "b:gb_bag5": -0.05294444725718581, "b:gb_l3": -0.01371064499052015, "b:gb_ohk": -0.04820480588166873, "b:gbc_bag3": -0.009978780226866327, "b:gbe_bag3": -0.0292470794008713, "b:gbn_bag5_w128": -0.0029032513594090493, "b:gbn_bag5_w512": 0.008589415226911652, "b:gbr_bag3": -0.01932634242624205, "b:gbx_bag3": -0.02473631487984731, "b:gbx_bag5": 0.0026061291977175307, "b:glm_dig": 0.03778785355347661, "b:glm_dl": -0.026265239163621588, "b:glm_dl_bag": 0.007619570732892796, "b:glm_dl_s50": -0.01589057953646802, "b:glm_dl_s50_bag": 0.020434071266557, "b:glm_ohk": -0.021658272129978732, "b:glm_ohk2": -0.0196684837560941, "b:glm_ohk2_bag": -0.042471963955860524, "b:glm_ohk_c01": 0.10592788348917859, "b:glm_ohkl": 0.04223275649409235, "b:glm_ohkl_bag": 0.040925149011110606, "b:glm_sb": 0.015782425613472196, "b:glm_te": -0.005184436085503802, "b:lgb_dig_l4": -0.027891674500056363, "b:lgb_dig_l7": -0.02186412655409438, "b:lgb_dig_s100": 0.03589010486874812, "b:lgb_dig_s100_bag": 0.010859780882435823, "b:lgb_dig_s100_bag5": -0.03728731587997563, "b:lgb_dig_s100_l3": 0.01423827047982986, "b:lgb_dig_s100_l3_bag": -0.006162448814395742, "b:lgb_diglead_bag": 0.007763582851059485, "b:lgb_diglead_bag5": 0.009117578055567227, "b:lgb_diglead_l2": 0.026382769433195005, "b:lgb_diglead_l3": -0.03945478160890352, "b:lgb_diglead_l3_bag": 0.0056052934586958625, "b:lgb_diglead_l4": 0.002456501679524635, "b:lgb_dk_l4": 0.0250852416583794, "b:lgb_raw_l31": 0.00981267380269437, "b:lgb_raw_l4": 0.02087321155263555, "b:lgb_raw_l4_cnt": -0.0209918008798821, "b:lgb_rawdig_l4": -0.02749188667334588, "b:lgb_te_l4": -0.006515188723849069, "b:lgb_te_l4_s100": -0.020994037647683976, "b:lgb_te_l4_s300": -0.01242837437677408, "b:lgb_te_l4_s5": -0.007472350724748886, "b:realmlp_allcat": -0.0072834856178715275, "b:realmlp_cnt": 0.0002164152740165542, "b:realmlp_dig": 0.012897170670949538, "b:realmlp_dig_s100": 0.000565203128950287, "b:realmlp_dig_s2": 0.0009704389429282402, "b:realmlp_rawdig": 0.009120429407365276, "b:realmlp_te": -0.02017974508103666, "b:realmlp_te_cnt": 0.01595553848213103, "b:realmlp_te_s2": 0.035416947477365866, "b:realmlp_v1": 0.017830419228308213, "b:xgb_dig_d2": 0.020832318644034946, "b:xgb_raw_d3": -0.02876669926417868, "b:xgb_te_d2": -0.007623174327739904, "x:Sold": 0.4752309053500493}'''
# Version 9: step 12, the same 88 members, from the step 11 file
STEP12_JSON = '''{"s:megayak_5f_lm": -0.02468300405897302, "s:masaya_gpt2": -0.04454170749225556, "s:lm7_5fold": 0.005025570671919622, "s:gpt2_seed2026": -0.004319757791790192, "s:tabpfn_slimypunk_tp1": 0.005373392488988152, "s:sadam_tabpfn_100k_tp3": -0.006034457751763471, "s:yekenot_realmlp": -0.016927539207871923, "s:alex_glm_gen": 0.0326359972931681, "s:daoviet_gpdvae": 0.008523141127242706, "s:gbn_bag5_w512": 0.010041144879994959, "s:sameer_glm10f": -0.04407672200988169, "s:sadam_v6_gb_xgb": -0.0013338744586116845, "s:sadam_v6_gb_lgb": -0.03883950449778915, "s:sadam_v4_cat": -0.011281822003238106, "s:gb_bag5": 0.006342651644130711, "s:megayak_shallow_tree": -0.00023072740159713759, "wide": 0.0104250785097767, "b:ebm_all_bag": 0.020184324578603533, "b:ebm_dig": 0.00166992502766065, "b:ebm_dig_bag": 0.01723750018475145, "b:ebm_dig_r7": -0.010548944173571157, "b:ebm_diglead_bag": -0.010244941036831325, "b:ebm_diglead_i25": -0.019923102272313364, "b:ebm_diglead_r11": 0.0008575136476480189, "b:ebm_diglead_r13": -0.019935799081965813, "b:ebm_i25_bag": 0.016149108537306375, "b:ebm_i25_bag4": -0.02175180803411238, "b:ebm_lead4_bag": 0.02219458919752827, "b:gb_bag10": -0.014885061335104548, "b:gb_bag5": -0.02472742789684929, "b:gb_l3": 0.003734734304943618, "b:gb_ohk": -0.013358971099544745, "b:gbc_bag3": 0.007961065421387413, "b:gbe_bag3": -0.015271351923606654, "b:gbn_bag5_w128": -0.008933655586128764, "b:gbn_bag5_w512": -0.010758744034910839, "b:gbr_bag3": -0.006955123794988345, "b:gbx_bag3": -0.010397998389884642, "b:gbx_bag5": 0.0013733213110612473, "b:glm_dig": 0.010469715929565953, "b:glm_dl": 0.005698571172044539, "b:glm_dl_bag": -0.010912091695915229, "b:glm_dl_s50": -0.0009263743743089046, "b:glm_dl_s50_bag": 0.006285482876699541, "b:glm_ohk": -0.017792983306962246, "b:glm_ohk2": -0.026553352411754817, "b:glm_ohk2_bag": 0.018063509349417993, "b:glm_ohk_c01": 0.0518313014124387, "b:glm_ohkl": 0.0147502183031705, "b:glm_ohkl_bag": 0.013009494413605145, "b:glm_sb": 0.003014623804522846, "b:glm_te": -0.00016288354602167536, "b:lgb_dig_l4": -0.018084508402080406, "b:lgb_dig_l7": -0.014216115754878815, "b:lgb_dig_s100": 0.01793669897573955, "b:lgb_dig_s100_bag": 0.006797783920993535, "b:lgb_dig_s100_bag5": -0.016226768818856024, "b:lgb_dig_s100_l3": -0.003720844821561325, "b:lgb_dig_s100_l3_bag": -0.0019144281445997997, "b:lgb_diglead_bag": 0.004701882765894067, "b:lgb_diglead_bag5": 0.015699215291609378, "b:lgb_diglead_l2": 0.02323499596333667, "b:lgb_diglead_l3": -0.024403734955726602, "b:lgb_diglead_l3_bag": 0.006528635145683499, "b:lgb_diglead_l4": -0.0018930204323077654, "b:lgb_dk_l4": 0.007000389297650995, "b:lgb_raw_l31": -0.00015425722083158653, "b:lgb_raw_l4": 0.012675928633482667, "b:lgb_raw_l4_cnt": 0.005559612013228819, "b:lgb_rawdig_l4": -0.015160100706178334, "b:lgb_te_l4": -0.005355769594532017, "b:lgb_te_l4_s100": -0.01105769248737589, "b:lgb_te_l4_s300": -0.005656078749547945, "b:lgb_te_l4_s5": 0.0041169061721358004, "b:realmlp_allcat": -0.0069676880090210344, "b:realmlp_cnt": -0.0029607318570637996, "b:realmlp_dig": 0.008024221742841663, "b:realmlp_dig_s100": -0.0006226256878995438, "b:realmlp_dig_s2": -0.0004038859525604855, "b:realmlp_rawdig": 0.01094427187659951, "b:realmlp_te": -0.005473379648691275, "b:realmlp_te_cnt": 0.004235836561586966, "b:realmlp_te_s2": 0.01785823291843117, "b:realmlp_v1": -0.0003186639173704878, "b:xgb_dig_d2": 0.012489789383744657, "b:xgb_raw_d3": -0.0077706813522084636, "b:xgb_te_d2": 0.0005314433562439851, "x:Sold": 0.11969368705778355}'''
# Version 10: the member step inside step 13, from the step 12 file
STEP13_JSON = '''{"s:megayak_5f_lm": -0.02270781232555902, "s:masaya_gpt2": -0.04263635979376339, "s:lm7_5fold": 0.0012279843232837013, "s:gpt2_seed2026": -0.009965130862174506, "s:tabpfn_slimypunk_tp1": 0.006698012013563353, "s:sadam_tabpfn_100k_tp3": 1.7059020627554616e-06, "s:yekenot_realmlp": -0.018822093253493846, "s:alex_glm_gen": 0.03839162898932659, "s:daoviet_gpdvae": 0.009438436785435112, "s:gbn_bag5_w512": 0.005808494830316598, "s:sameer_glm10f": -0.05582213663453058, "s:sadam_v6_gb_xgb": 0.009375148913737965, "s:sadam_v6_gb_lgb": -0.05098261702067282, "s:sadam_v4_cat": -0.013494430362265025, "s:gb_bag5": 0.014820553382454851, "s:megayak_shallow_tree": -0.0028111147811287415, "wide": 0.00943322970874392, "b:ebm_all_bag": 0.03362027136891233, "b:ebm_dig": 0.004028235614848501, "b:ebm_dig_bag": 0.015107234323490981, "b:ebm_dig_r7": -0.015250527621541053, "b:ebm_diglead_bag": 0.008710929036152408, "b:ebm_diglead_i25": -0.020047985738244848, "b:ebm_diglead_r11": -0.01174064059888286, "b:ebm_diglead_r13": -0.03777530237156378, "b:ebm_i25_bag": 0.009315601751082358, "b:ebm_i25_bag4": -0.027248315245499123, "b:ebm_lead4_bag": 0.03382418451374998, "b:gb_bag10": -0.015226252051252425, "b:gb_bag5": -0.042668702583547255, "b:gb_l3": 0.008738585111481248, "b:gb_ohk": -0.009086710844545738, "b:gbc_bag3": 0.014938143414460913, "b:gbe_bag3": -0.017722229334826523, "b:gbn_bag5_w128": -0.007460387124433427, "b:gbn_bag5_w512": -0.022272098874120152, "b:gbr_bag3": -0.007705849448279651, "b:gbx_bag3": -0.0059427740741583, "b:gbx_bag5": -0.003630504875517701, "b:glm_dig": 0.00929227575842708, "b:glm_dl": 0.011086650524119293, "b:glm_dl_bag": -0.006309298896601785, "b:glm_dl_s50": -0.008424132942630277, "b:glm_dl_s50_bag": 0.008766978253716947, "b:glm_ohk": -0.018492799899205555, "b:glm_ohk2": -0.024490531199399687, "b:glm_ohk2_bag": 0.029738235866735647, "b:glm_ohk_c01": 0.05616961813762058, "b:glm_ohkl": 0.0023689830672410897, "b:glm_ohkl_bag": 0.017641588235602634, "b:glm_sb": 0.0011128577831866775, "b:glm_te": -0.00021293835200713156, "b:lgb_dig_l4": -0.024472905376888354, "b:lgb_dig_l7": -0.01677426931622322, "b:lgb_dig_s100": 0.021927625163681956, "b:lgb_dig_s100_bag": 0.006272016369582174, "b:lgb_dig_s100_bag5": -0.01508877641505651, "b:lgb_dig_s100_l3": -0.012783904966123956, "b:lgb_dig_s100_l3_bag": 0.0004365322033999709, "b:lgb_diglead_bag": 0.007015676126699508, "b:lgb_diglead_bag5": 0.026423929231910632, "b:lgb_diglead_l2": 0.032955000611762826, "b:lgb_diglead_l3": -0.03040294726859985, "b:lgb_diglead_l3_bag": 0.005634831649085182, "b:lgb_diglead_l4": -0.004551738719740078, "b:lgb_dk_l4": 0.006370239855686273, "b:lgb_raw_l31": -0.0037932487417954228, "b:lgb_raw_l4": 0.01696285449417552, "b:lgb_raw_l4_cnt": 0.010604366654493923, "b:lgb_rawdig_l4": -0.02025490613654443, "b:lgb_te_l4": -0.010360007934525699, "b:lgb_te_l4_s100": -0.01431303256800578, "b:lgb_te_l4_s300": -0.0068353769234054135, "b:lgb_te_l4_s5": 0.008612488976405736, "b:realmlp_allcat": -0.010042412116887648, "b:realmlp_cnt": -0.005467215693223665, "b:realmlp_dig": 0.009762972846129666, "b:realmlp_dig_s100": -0.0016026257112236046, "b:realmlp_dig_s2": -0.0013842473149377546, "b:realmlp_rawdig": 0.016858636951778253, "b:realmlp_te": -0.0056002941181404475, "b:realmlp_te_cnt": 0.0033848023010744858, "b:realmlp_te_s2": 0.021684608739645936, "b:realmlp_v1": -0.0037476925030982874, "b:xgb_dig_d2": 0.016289703584232346, "b:xgb_raw_d3": -0.008894096595013723, "b:xgb_te_d2": 0.005996530520395943, "x:Sold": 0.13565833500400254}'''

import os, json, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import rankdata
from scipy.special import logit
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('KAGGLE_WORKING', '/kaggle/working'))

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def find(name, digest=None, inside=None):   # a file anywhere under /kaggle/input, checked by SHA-256
    hits = [p for p in sorted(INPUT.rglob(name)) if inside is None or inside in str(p)]
    if digest is not None:
        hits = [p for p in hits if sha256(p) == digest]
    assert hits, (name, digest, inside)
    return hits[0]

def U(v):   # rank fraction
    return (rankdata(v) - 0.5) / len(v)

def L(v):   # logit rank: AUC only sees order, so every input goes on one scale
    return logit(U(v))

SHA = {   # public inputs, each checked byte for byte
    'bundle':   ('rank1_q17_streams.npz', '0fe4ab1263a673c8065d7145590b92da440263161bf860ce45f2976b5c4fa56f'),    # Aman Atar: 0.95370 file + 16 streams
    'wide_oof': ('wide_lm_stack_nested.npy', '69375871698f1adf2b4c666fc7f5180edf00ac6836668353b8e7e4ca43eb2722'), # our wide LM stack, nested OOF
    'wide_te':  ('wide_lm_stack_test.npy', 'b37471e6c34a50715284e807d68e15e5f3ad033e72400904f27876b24041f8cc'),   # our wide LM stack, test
    'l7f_te':   ('lm7_test_5fold.npy', '68639fec79f081e97d40185443fa3903c9db3e62698abb01dfe31dcee217462a'),       # our LM7, 5 fold models on test
    'lm_te':    ('megalm_test_sha3168.npy', '3168779be644a530c6bdc4c141cae168bfc2feb01ac21090c917cc978a7687d3'),  # megayak LM, test (frozen copy)
    'g26_te':   ('gpt2_s2026_test_5fold.npy', 'cea5f578ddf9118d7ea47ea38e2591ea66102ab2ce9aff37f5586c81adfbaa0e'),    # GPT-2 seed 2026, 5 fold models on test
    'bank':     'ddd0b8e8112d49846e415a4ddbead968bce6d50fb8f351785f21a8053f420538',                                 # our OOF bank: digest of its 140 files
    'S':        ('alexchilton_v23_submission.csv', '36686835b6ca3bd805ef578fc50cd62b3cbbe4d2a70275e50d6c38c072d4ad77'),  # alexchilton's shallow-tree stack (pinned copy)
    'A':        ('asteriospanoras_v11_submission.csv', '9ecd6ea1ed92bfaa69e609d5ad2d95f039d7728e1b797f4b537a948399c463fc'),  # Asterios Panoras v11, public 0.95422
    'O':        ('areebali_v22_submission.csv', 'b08485cffab7f66cac99ff7872c8d1d1563128c6d63d88f26dc72452feb1ce14'),  # Areeb Ali v22, public 0.95421
    'B':        ('bhuvannr_step10_submission.csv', 'ce020d84c6279f5946f94983db6fe68b96a0dd96b5809d0a96ecd263fe586703'),  # BhuvanNR step 10, public 0.95426
}

Z = np.load(find(*SHA['bundle']), allow_pickle=False)
y, FOLD = Z['y_train'].astype(int), Z['folds']
NAMES = [str(s) for s in Z['stream_names']]                            # the 16 public models in the bundle
auc = lambda s: roc_auc_score(y, s)
one = lambda X: np.column_stack([X, np.ones(len(X))])
XO = np.column_stack([L(Z['oof_' + n]) for n in NAMES])                # their OOF predictions, as logit ranks
XT = np.column_stack([L(Z['test_' + n]) for n in NAMES])               # their test predictions
AT = L(Z['test_anchor_095370'])                                        # the public 0.95370 file (Aman Atar)
W_OOF, W_T = L(np.load(find(*SHA['wide_oof']))), L(np.load(find(*SHA['wide_te'])))   # our wide LM stack: nested OOF, test

def replay(T, O):   # fit the 0.95370 file on the test rows, apply the same weights to the OOF predictions
    b = np.linalg.lstsq(one(T), AT, rcond=None)[0]
    return one(O) @ b, 1 - ((AT - one(T) @ b) ** 2).mean() / AT.var()
A16, R2_16 = replay(XT, XO)                                            # version 3: on the 16 public models
A_OOF, R2 = replay(np.column_stack([XT, W_T]), np.column_stack([XO, W_OOF]))   # version 4: 16 models + our wide stack

def change(v, base):                                                   # (AUC change x 1e5, folds up out of 5)
    up = sum(roc_auc_score(y[FOLD == f], v[FOLD == f]) > roc_auc_score(y[FOLD == f], base[FOLD == f]) for f in range(5))
    return 1e5 * (auc(v) - auc(base)), int(up)

rng = np.random.default_rng(0)
NOISE = {}                                                             # replayed file minus our wide stack on random samples of rows
for n in (60_000, 240_000):                                            # the sizes of the public and private parts of the test set
    g = []
    for _ in range(100):
        i = rng.choice(len(y), n, replace=False)
        g.append(roc_auc_score(y[i], A_OOF[i]) - roc_auc_score(y[i], W_OOF[i]))
    NOISE[n] = (1e5 * np.mean(g), 1e5 * np.std(g))

# The version 3 file: the 0.95370 file with 0.06 of megayak's LM moved to our 5-fold LM. Public 0.95370. Every mix starts here.
REF_OOF = A_OOF + 0.06 * (XO[:, NAMES.index('lm7_5fold')] - XO[:, NAMES.index('megayak_5f_lm')])
U0 = U(AT + 0.06 * (L(np.load(find(*SHA['l7f_te']))) - L(np.load(find(*SHA['lm_te'])))))

# Members a mix can move towards: the 16 public models, our wide stack, 70 models of our OOF bank.
BANK = find('manifest.csv', inside='oof-bank').parent
bank = sorted(p for p in BANK.glob('*.npy') if p.name.startswith(('oof_', 'test_')))
assert hashlib.sha256(''.join(sha256(p) for p in bank).encode()).hexdigest() == SHA['bank'], 'the OOF bank changed'

def member(k, part):                                                   # part is 'oof' or 'test'; returned as a logit rank
    if k.startswith('s:'): return L(Z[f'{part}_{k[2:]}'])
    if k.startswith('b:'): return L(np.load(BANK / f'{part}_{k[2:]}.npy'))
    return W_OOF if part == 'oof' else W_T

RECIPES = json.loads(RECIPE_JSON)
MIX_T = {r: (1 - sum(w.values())) * logit(U0) for r, w in RECIPES.items()}     # test side: what gets submitted
MIX_O = {r: (1 - sum(w.values())) * REF_OOF for r, w in RECIPES.items()}       # OOF side: the same weights on OOF predictions
STEPS = json.loads(STEP_JSON)                                                  # version 5: three steps from the 0.95388 file (below)
SUM_T, SUM_O = {r: 0.0 for r in STEPS}, {r: 0.0 for r in STEPS}
for k in RECIPES['bold']:
    t, o = member(k, 'test'), member(k, 'oof')
    for r, w in RECIPES.items():
        MIX_T[r] = MIX_T[r] + w[k] * t
        MIX_O[r] = MIX_O[r] + w[k] * o
    for r, w in STEPS.items():
        SUM_T[r] = SUM_T[r] + w[k] * t
        SUM_O[r] = SUM_O[r] + w[k] * o
CV = {r: change(MIX_O[r], REF_OOF) for r in RECIPES}                   # in-sample: these weights saw all five folds
WIDE_CV = {w: change((1 - w) * A_OOF + w * W_OOF, A_OOF)[0] for w in (0.10, 0.25)}

# Version 5. Areeb Ali extrapolated the version 4 mixes: bold + 0.5 (bold - balanced) scored 0.95388.
def comb(M, ref, a, b=0.0, c=0.0):                                     # a x bold + b x balanced + c x careful, as weights on the members
    return a * M['bold'] + b * M['balanced'] + c * M['careful'] + (1 - a - b - c) * ref
P4_T, P4_O = comb(MIX_T, logit(U0), 1.5, -0.5), comb(MIX_O, REF_OOF, 1.5, -0.5)
# Our steps. Each starts from an earlier file, which goes in as a logit rank, and moves towards the same 87 members.
FROM = {'joint': 'areeb', 'cv_led': 'areeb', 'step1': 'areeb', 'step2_half': 'step1', 'step2': 'step1', 'step3': 'step2', 'step4': 'step3', 'step5': 'step4', 'step6': 'step4', 'step7': 'step6', 'step8': 'step7', 'step9': 'step7'}
STEP_T, STEP_O = {'areeb': P4_T}, {'areeb': P4_O}
for r, w in STEPS.items():
    STEP_T[r] = (1 - sum(w.values())) * L(STEP_T[FROM[r]]) + SUM_T[r]  # test side: what gets submitted
    STEP_O[r] = (1 - sum(w.values())) * L(STEP_O[FROM[r]]) + SUM_O[r]  # OOF side: the same weights on OOF predictions
CV5 = {'areeb': change(P4_O, MIX_O['bold'])}                           # in-sample change of each file from the file it starts from
CV5.update({r: change(STEP_O[r], STEP_O[FROM[r]]) for r in STEPS})
CVB = {r: change(STEP_O[r], MIX_O['bold']) for r in STEP_O}            # and from the version 4 file

print(f'the 0.95370 file replayed on OOF | 16 public models (v3): R² {R2_16:.4f}, AUC {auc(A16):.6f}'
      f' | 16 models + our wide stack (v4): R² {R2:.4f}, AUC {auc(A_OOF):.6f}')
print(f'our wide LM stack, nested: {auc(W_OOF):.6f}')
for n, (m, s) in NOISE.items():
    print(f'gap (replayed file - wide stack) on {n:,} random rows: {m:+.1f} ± {s:.1f} points')

In [ ]:
# The three mixes on OOF. These weights saw all five folds, so this is the in-sample change;
# the nested figures in the table above come from refitting the weights on four folds at a time.
NESTED = {'careful': 22.9, 'balanced': 14.3, 'bold': 19.5}
print(f"{'mix':10s} {'OOF AUC':>9s} {'in-sample':>10s} {'folds up':>9s} {'nested':>8s}")
print(f"{'v3 file':10s} {auc(REF_OOF):9.6f}")
for r in ('careful', 'balanced', 'bold'):
    print(f'{r:10s} {auc(MIX_O[r]):9.6f} {CV[r][0]:+9.1f} {CV[r][1]:>7d}/5 {NESTED[r]:+8.1f}')
w = pd.Series(RECIPES['bold']).sort_values(key=abs, ascending=False)
print(f'\nbold mix: {len(w)} members, weights sum to {w.sum():+.3f}; the twelve largest')
print(w.head(12).round(3).to_string())

In [ ]:
# Version 5 on OOF. Areeb Ali's eight probes are combinations of the three mixes, so their out-of-fold change is one line each.
AREEB_W = {'1.25 x bold': (1.25, 0, 0, 0.95387), '1.5 x bold': (1.5, 0, 0, 0.95387), 'bold + 0.5 (bold - careful)': (1.5, 0, -0.5, 0.95387),
           'bold + 0.5 (bold - balanced)': (1.5, -0.5, 0, 0.95388), 'bold + 1.0 (bold - balanced)': (2.0, -1.0, 0, 0.95387),
           'bold + 1.5 (bold - balanced)': (2.5, -1.5, 0, 0.95387), 'bold + 2.0 (bold - balanced)': (3.0, -2.0, 0, 0.95385),
           '1.25 x (bold + 1.0 (bold - balanced))': (2.5, -1.25, 0, 0.95386)}
print(f"{'his probe (scores from his notebook)':40s} {'OOF change from bold':>21s} {'folds up':>9s} {'public LB':>10s}")
for name, (a, b, c, lb) in AREEB_W.items():
    ch = change(comb(MIX_O, REF_OOF, a, b, c), MIX_O['bold'])
    print(f'{name:40s} {ch[0]:+21.1f} {ch[1]:>7d}/5 {lb:10.5f}')

# Our twelve steps. In-sample again (these weights saw all five folds); the nested figures are in the tables at the top.
NESTED5 = {'joint': 3.3, 'cv_led': 7.3, 'step1': -1.1, 'step2_half': -8.6, 'step2': -18.4, 'step3': -29.2, 'step4': -18.6, 'step5': -33.3, 'step6': 16.0, 'step7': -19.0, 'step8': 9.6, 'step9': 7.7}
LB5 = {'joint': 0.95388, 'cv_led': 0.95387, 'step1': 0.95390, 'step2_half': 0.95393, 'step2': 0.95396, 'step3': 0.95404, 'step4': 0.95408, 'step5': 0.95407, 'step6': 0.95412, 'step7': 0.95417, 'step8': 0.95418, 'step9': 0.95421}
print(f"\n{'file':12s} {'starts from':>12s} {'OOF AUC':>9s} {'in-sample':>10s} {'folds up':>9s} {'nested':>8s} {'from v4':>8s} {'public LB':>10s}")
print(f"{'v4 (bold)':12s} {'':>12s} {auc(MIX_O['bold']):9.6f} {'':>10s} {'':>9s} {'':>8s} {0:+8.1f} {0.95386:10.5f}")
print(f"{'areeb':12s} {'v4 (bold)':>12s} {auc(P4_O):9.6f} {CV5['areeb'][0]:+10.1f} {CV5['areeb'][1]:>7d}/5 {'':>8s} {CVB['areeb'][0]:+8.1f} {0.95388:10.5f}")
for r in STEPS:
    print(f'{r:12s} {FROM[r]:>12s} {auc(STEP_O[r]):9.6f} {CV5[r][0]:+10.1f} {CV5[r][1]:>7d}/5 {NESTED5[r]:+8.1f} {CVB[r][0]:+8.1f} {LB5[r]:10.5f}')
tot = pd.Series({k: sum(STEPS[r][k] for r in ('step1', 'step2', 'step3', 'step4', 'step6', 'step7', 'step9')) for k in STEPS['step4']}).sort_values(key=abs, ascending=False)
print('\nsteps 1 to 4, 6, 7 and 9 added up (the weights compound slightly, so this is a guide): the twelve largest moves')
print(tot.head(12).round(3).to_string())

In [ ]:
# Version 7. What the blend wants, measured on the CV-led file of version 5 (public 0.95387), and the CV-backed file that follows.
KEYS = list(RECIPES['bold']); MO = {k: member(k, 'oof') for k in KEYS}             # the 87 members, OOF side, as logit ranks
ref7 = L(STEP_O['cv_led']); single = {k: auc(MO[k]) for k in KEYS}
F7 = [FOLD == f for f in range(5)]; A0 = auc(ref7); F0 = np.array([roc_auc_score(y[i], ref7[i]) for i in F7])
def gain(s):                                                                       # (OOF change x 1e5 from the CV-led file, folds up)
    return 1e5 * (auc(s) - A0), int((np.array([roc_auc_score(y[i], s[i]) for i in F7]) > F0).sum())

# A. appetite: what a 2% move towards one member is worth to the blend
APP = {k: gain(ref7 + 0.02 * (MO[k] - ref7)) for k in KEYS}
print(f'reference: the CV-led file, OOF AUC {A0:.6f}, {len(KEYS)} members')
print('\nA. a 2% move towards one member: the eight largest gains and the four largest losses')
for k in sorted(KEYS, key=lambda k: -APP[k][0])[:8] + sorted(KEYS, key=lambda k: APP[k][0])[:4]:
    print(f'  {k:26s} {APP[k][0]:+6.2f}  folds up {APP[k][1]}/5   alone {single[k]:.6f}')
ap, sg = np.array([APP[k][0] for k in KEYS]), np.array([single[k] for k in KEYS])
print(f'  rank correlation between a member\'s own AUC and the gain of a move towards it: {np.corrcoef(rankdata(sg), rankdata(ap))[0, 1]:+.2f}')
print('  the four target-first language models: ' + ', '.join(f'{k[2:]} {APP[k][0]:+.2f} (alone {single[k]:.5f})' for k in ('s:gpt2_seed2026', 's:masaya_gpt2', 's:lm7_5fold', 's:megayak_5f_lm')))

# B. a stronger model of the same family: weight moved from the weaker one to the stronger one, best size on a small grid
FAM = [['lgb_dig_s100', 'lgb_dig_s100_bag', 'lgb_dig_s100_bag5'], ['lgb_dig_s100_l3', 'lgb_dig_s100_l3_bag'], ['lgb_diglead_bag', 'lgb_diglead_bag5'],
       ['lgb_diglead_l3', 'lgb_diglead_l3_bag'], ['ebm_dig', 'ebm_dig_r7', 'ebm_dig_bag'], ['ebm_i25_bag', 'ebm_i25_bag4'], ['gb_bag5', 'gb_bag10'],
       ['gbx_bag3', 'gbx_bag5'], ['glm_dl', 'glm_dl_bag'], ['glm_dl_s50', 'glm_dl_s50_bag'], ['glm_ohk2', 'glm_ohk2_bag'], ['glm_ohkl', 'glm_ohkl_bag'],
       ['realmlp_dig', 'realmlp_dig_s100'], ['realmlp_te', 'realmlp_te_s2'], ['lgb_te_l4_s5', 'lgb_te_l4', 'lgb_te_l4_s100', 'lgb_te_l4_s300'],
       ['realmlp_dig', 'realmlp_rawdig'], ['lgb_raw_l4', 'lgb_raw_l31'], ['xgb_dig_d2', 'lgb_dig_l4'], ['lgb_dig_l4', 'lgb_dig_l7']]
PAIRS = []
for mem in FAM:
    mem = sorted(['b:' + k for k in mem if 'b:' + k in MO], key=lambda k: single[k])
    for w in mem[:-1]:
        (g, up), d = max((gain(ref7 + d * (MO[mem[-1]] - MO[w])), d) for d in (0.01, 0.02, 0.04, 0.07, 0.10))
        PAIRS.append((w[2:], mem[-1][2:], 1e5 * (single[mem[-1]] - single[w]), g, up, d))
print('\nB. same family, weaker -> stronger: gain alone | best gain in the blend, folds up, size of the move')
for w, st, ds, g, up, d in PAIRS:
    print(f'  {w:20s} -> {st:20s} alone {ds:+7.1f} | in the blend {g:+6.2f}  {up}/5  d {d:.2f}')
BIG = np.array([(ds, g) for _, _, ds, g, _, _ in PAIRS if ds >= 5])
print(f'  {len(PAIRS)} pairs; the {len(BIG)} with a gain of 5 or more alone: mean alone {BIG[:, 0].mean():+.1f}, mean in the blend {BIG[:, 1].mean():+.2f}'
      f' -> the blend keeps {100 * BIG[:, 1].sum() / BIG[:, 0].sum():.1f}% of it')

# C. the step: LM weight from megayak's compact LM to the average of the two target-first GPT-2 seeds
dO7 = (MO['s:masaya_gpt2'] + MO['s:gpt2_seed2026']) / 2 - MO['s:megayak_5f_lm']
A7 = (0.0, 0.05, 0.10, 0.15, 0.20, 0.25)
TF = {a: np.array([roc_auc_score(y[i], (ref7 + a * dO7)[i]) for i in F7]) for a in A7}
GA = {a: 1e5 * (auc(ref7 + a * dO7) - A0) for a in A7}
print('\nC. weight a moved from megayak\'s LM to the two GPT-2 seeds (alone: megayak '
      f"{single['s:megayak_5f_lm']:.5f}, masayakawamata's GPT-2 {single['s:masaya_gpt2']:.5f}, seed 2026 {single['s:gpt2_seed2026']:.5f})")
for a in A7[1:]:
    print(f'  a {a:.2f}: OOF {GA[a]:+5.1f}  folds up {int((TF[a] > TF[0.0]).sum())}/5')
PICK = [max(A7, key=lambda a: np.delete(TF[a] - TF[0.0], f).mean()) for f in range(5)]   # a chosen on four folds ...
NEST7 = [1e5 * (TF[PICK[f]][f] - TF[0.0][f]) for f in range(5)]                          # ... and scored on the fifth
a7 = max((a for a in A7[1:] if (TF[a] > TF[0.0]).all()), key=lambda a: (TF[a] - TF[0.0]).mean())   # the best move that is up in all five folds
K1_O = ref7 + a7 * dO7
print(f'  nested (a chosen on four folds, scored on the fifth): picks {PICK} -> {np.mean(NEST7):+.1f}, folds up {sum(v > 0 for v in NEST7)}/5')
print(f'  taken: a = {a7:.2f} -> OOF AUC {auc(K1_O):.6f} ({1e5 * (auc(K1_O) - A0):+.1f} from the CV-led file)')

# Test side. The CV-led file is rebuilt with the five-fold test predictions of GPT-2 seed 2026 (the bundle carries the fold-0
# model's): once as a member of our mixes, once inside the 0.95370 file through its fitted weight there. OOF is unchanged by this.
T0, T5 = L(Z['test_gpt2_seed2026']), L(np.load(find(*SHA['g26_te'])))
bA = np.linalg.lstsq(one(np.column_stack([XT, W_T])), AT, rcond=None)[0]
MT5 = {k: (T5 if k == 's:gpt2_seed2026' else member(k, 'test')) for k in KEYS}
def mix5(w, base):
    s = (1 - sum(w.values())) * base
    for k in KEYS: s = s + w[k] * MT5[k]
    return s
ref5 = logit(U(AT + bA[NAMES.index('gpt2_seed2026')] * (T5 - T0) + 0.06 * (L(np.load(find(*SHA['l7f_te']))) - L(np.load(find(*SHA['lm_te']))))))
p4_5 = 1.5 * mix5(RECIPES['bold'], ref5) - 0.5 * mix5(RECIPES['balanced'], ref5)
CVLED5_T = mix5(STEPS['cv_led'], L(p4_5))
K1_T = L(CVLED5_T) + a7 * ((L(Z['test_masaya_gpt2']) + T5) / 2 - L(Z['test_megayak_5f_lm']))
rk = lambda a, b: np.corrcoef(rankdata(a), rankdata(b))[0, 1]
print(f'\ntest side: rank correlation of the two seed 2026 test files {rk(T0, T5):.5f}; the CV-led file with either {rk(CVLED5_T, STEP_T["cv_led"]):.6f}; '
      f'the new CV-backed file against the CV-led file {rk(K1_T, STEP_T["cv_led"]):.6f}')

fig = plt.figure(figsize=(16, 6.2), dpi=110)
ax = fig.add_axes([0.065, 0.13, 0.42, 0.66])
LMS = {'s:megayak_5f_lm': ("megayak's compact LM", (10, -3), 'left'), 's:masaya_gpt2': ("masayakawamata's GPT-2", (-10, -2), 'right'),
       's:gpt2_seed2026': ('GPT-2 seed 2026 (ours)', (10, 1), 'left'), 's:lm7_5fold': ('our compact LM', (-10, 2), 'right')}
oth = [k for k in KEYS if k not in LMS]
ax.axhline(0, color=AXIS, lw=1, zorder=1)
ax.scatter([single[k] for k in oth], [APP[k][0] for k in oth], s=46, color=MUTED, edgecolor=SURFACE, linewidth=1, zorder=3)
for k, (lab, off, ha) in LMS.items():
    ax.scatter([single[k]], [APP[k][0]], s=110, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=5)
    ax.annotate(lab, (single[k], APP[k][0]), xytext=off, textcoords='offset points', fontsize=9.5, color=INK, ha=ha, va='center')
ax.set_xlim(max(0.940, sg.min() - 0.0004), sg.max() + 0.0006)
ax.set_xlabel('out-of-fold AUC of the member alone'); ax.set_ylabel('gain of a 2% move towards it, points')
ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[plt.Line2D([], [], color=BLUE, marker='o', lw=0, ms=9, label='the four target-first language models'),
                   plt.Line2D([], [], color=MUTED, marker='o', lw=0, ms=7, label='the other 83 members')], loc='lower right', frameon=False, fontsize=9.5)
head(ax, 'The blend asks for the GPT-2 seeds, not for the stronger compact LM', 'each dot is one member: its own out-of-fold AUC against what a 2% move towards it adds to the CV-led file')
ax = fig.add_axes([0.57, 0.13, 0.40, 0.66])
xs = np.array(A7); ax.axhline(0, color=AXIS, lw=1, zorder=1)
for f in range(5):
    ax.plot(xs, [1e5 * (TF[a][f] - TF[0.0][f]) for a in A7], color=MUTED, lw=1, alpha=0.8, zorder=2)
ax.plot(xs, [GA[a] for a in A7], color=BLUE, lw=2.5, marker='o', ms=9, mec=SURFACE, mew=1.5, zorder=5)
ax.plot([a7], [GA[a7]], marker='o', ms=13, mfc=BLUE, mec=INK, mew=1.8, zorder=6)
ax.annotate(f'taken: {GA[a7]:+.1f}, up in all five folds', (a7, GA[a7]), xytext=(-10, 12), textcoords='offset points', fontsize=9.5, color=INK, ha='right', va='bottom', fontweight='bold')
ax.set_xticks(xs); ax.set_xlabel("weight moved from megayak's LM to the average of the two GPT-2 seeds"); ax.set_ylabel('out-of-fold AUC change, points')
ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[plt.Line2D([], [], color=BLUE, marker='o', lw=2.5, ms=8, mec=SURFACE, label='all 700,000 rows'),
                   plt.Line2D([], [], color=MUTED, lw=1, label='each of the five folds')], loc='lower right', frameon=False, fontsize=9.5)
head(ax, 'The step: language-model weight moves to the GPT-2 seeds', f'nested: {np.mean(NEST7):+.1f} points with the size chosen on four folds and scored on the fifth')
plt.show()

In [ ]:
# Version 8. Step 10 = the 0.95421 file + a public-model step over the 87 members and alexchilton's stack file;
# step 11 = step 10 + 1.0 x (Asterios Panoras' 0.95422 file - Areeb Ali's 0.95421 file). Pinned public copies, checked by SHA-256.
S_T, A_T, O_T = (L(pd.read_csv(find(*SHA[k]))['is_fraud'].to_numpy()) for k in ('S', 'A', 'O'))
KEYS = list(RECIPES['bold']); XT87 = one(np.column_stack([member(k, 'test') for k in KEYS])); XO87 = one(np.column_stack([member(k, 'oof') for k in KEYS]))
def replay87(t):                                                   # fit a public file on the test rows with the 87 members; same weights on OOF
    b = np.linalg.lstsq(XT87, t, rcond=None)[0]
    return XO87 @ b, 1 - ((t - XT87 @ b) ** 2).mean() / t.var()
S_O, r2S = replay87(S_T); A_O, r2A = replay87(A_T); O_O, r2O = replay87(O_T)
print(f'replay on the 87 members, R2 on the test rows: stack {r2S:.4f} (OOF AUC {auc(S_O):.6f}), A {r2A:.4f}, O {r2O:.4f}')
W10 = json.loads(STEP10_JSON)
X10_T = (1 - sum(W10.values())) * L(STEP_T['step9']) + sum(w * (S_T if k == 'x:Sold' else member(k, 'test')) for k, w in W10.items())
X10_O = (1 - sum(W10.values())) * L(STEP_O['step9']) + sum(w * (S_O if k == 'x:Sold' else member(k, 'oof')) for k, w in W10.items())
X11_T, X11_O = L(X10_T) + 1.0 * (A_T - O_T), L(X10_O) + 1.0 * (A_O - O_O)
print(f"step 10: weight on the stack {W10['x:Sold']:+.3f}, total moved {sum(W10.values()):+.3f}")
for name, (o, base) in {'step 10 from step 9': (X10_O, STEP_O['step9']), 'step 11 from step 10': (X11_O, X10_O),
                        'step 11 from step 9': (X11_O, STEP_O['step9'])}.items():
    ch = change(o, base); print(f'{name:22s} OOF change (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5')
w = pd.Series(W10).sort_values(key=abs, ascending=False)
print('\nstep 10, the ten largest moves'); print(w.head(10).round(3).to_string())

In [ ]:
# Version 9. Step 12 = step 11 + a public-model step over the same 88 members (refitted with the two 0.95428 files of 17 Nov)
# + 0.3 x (BhuvanNR's step-10 file B - step 11) + 0.5 x (A - O). B is read from the pinned copy, checked by SHA-256.
B_T = L(pd.read_csv(find(*SHA['B']))['is_fraud'].to_numpy()); B_O, r2B = replay87(B_T)
G11_T, G11_O = L(X11_T), L(X11_O)
W12 = json.loads(STEP12_JSON)
Z12_T = (1 - sum(W12.values())) * G11_T + sum(w * (S_T if k == 'x:Sold' else member(k, 'test')) for k, w in W12.items())
Z12_O = (1 - sum(W12.values())) * G11_O + sum(w * (S_O if k == 'x:Sold' else member(k, 'oof')) for k, w in W12.items())
X12_T = L(Z12_T) + 0.3 * (B_T - G11_T) + 0.5 * (A_T - O_T)
X12_O = L(Z12_O) + 0.3 * (B_O - G11_O) + 0.5 * (A_O - O_O)
print(f'B replayed on the 87 members: R2 {r2B:.4f}, OOF AUC {auc(B_O):.6f}')
print(f"step 12: weight on the stack {W12['x:Sold']:+.3f}, total moved {sum(W12.values()):+.3f}")
for name, (o, base) in {'member step from step 11': (Z12_O, X11_O), 'step 12 from step 11': (X12_O, X11_O),
                        'step 12 from step 9': (X12_O, STEP_O['step9'])}.items():
    ch = change(o, base); print(f'{name:26s} OOF change (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5')
w = pd.Series(W12).sort_values(key=abs, ascending=False)
print('\nstep 12, the ten largest member moves'); print(w.head(10).round(3).to_string())

In [ ]:
# Version 10. Step 13 = step 12 + a member step over the same 88 members (model refitted with the two files of 17 Nov)
# + 1.0 x r, where r is the part of the last step (step 12 - step 11) that does not lie along (B - step 11): Areeb Ali's split.
P13_T, G13_T, P13_O, G13_O = L(X12_T), L(X11_T), L(X12_O), L(X11_O)
W13 = json.loads(STEP13_JSON)
Q13_T = (1 - sum(W13.values())) * P13_T + sum(w * (S_T if k == 'x:Sold' else member(k, 'test')) for k, w in W13.items())
Q13_O = (1 - sum(W13.values())) * P13_O + sum(w * (S_O if k == 'x:Sold' else member(k, 'oof')) for k, w in W13.items())
d13, e13 = P13_T - G13_T, B_T - G13_T
beta13 = float(d13 @ e13 / (e13 @ e13))                                 # least squares on the test rows, no labels
R13_T = d13 - beta13 * e13
R13_O = (P13_O - G13_O) - beta13 * (B_O - G13_O)
X13_T = L(Q13_T) + R13_T
X13_O = L(Q13_O) + R13_O
print(f'split of the last step: {beta13:.3f} x (B - step 11) + r; correlation of the two parts {np.corrcoef(d13, e13)[0, 1]:.3f}')
for name, (o, base) in {'member step from step 12': (Q13_O, X12_O), 'step 13 from step 12': (X13_O, X12_O),
                        'step 13 from step 9': (X13_O, STEP_O['step9'])}.items():
    ch = change(o, base); print(f'{name:26s} OOF change (in-sample, replay) {ch[0]:+6.1f}  folds up {ch[1]}/5')
w = pd.Series(W13).sort_values(key=abs, ascending=False)
print('\nstep 13, the ten largest member moves'); print(w.head(10).round(3).to_string())

In [ ]:
# The files. submission.csv is step 13 (public 0.95432): the version 5 file (tuned on the public rows) plus the steps of versions 6, 8, 9 and 10.
sample = pd.read_csv(find('sample_submission.csv', inside='playground-series-s6e11'))
r = lambda a, b: np.corrcoef(rankdata(a), rankdata(b))[0, 1]   # rank correlation
OUTS = {'submission.csv': X13_T,                      # public 0.95432 (step 13)
        'submission_cv.csv': K1_T,                     # public 0.95379: the CV-backed file of version 7, our best out-of-fold AUC
        'submission_step12.csv': X12_T,                # public 0.95430
        'submission_step11.csv': X11_T,                # public 0.95427
        'submission_step10.csv': X10_T,                # public 0.95426
        'submission_bold.csv': MIX_T['bold']}          # version 4, public 0.95386
for name, t in OUTS.items():
    sample[['id']].assign(is_fraud=U(t)).to_csv(WORK / name, index=False)
    print(f'{name:22s} rank correlation with the 0.95370 file {r(t, AT):.6f}, with the 0.95388 file {r(t, P4_T):.6f}')
pd.read_csv(WORK / 'submission.csv').head()

## A correction to version 3 (from version 4)

Version 3 replayed the 0.95370 file on out-of-fold rows with the 16 public models alone (98.3% of the file) and reported
**0.95186**, 67 ± 34 points below our wide LM stack on a public-sized sample. The part those 16 models leave out turns out to point
the same way as our wide stack (correlation +0.63). Adding the wide stack to the replay explains 98.95% of the file and puts it at
**0.95204**: about 54 ± 32 points below the wide stack on 60,000 rows, 55 ± 12 on 240,000. Version 3 was too hard on the file by
17 points. Its conclusion stands (the public board shows +72, well outside either range), and every CV change here is
measured from the corrected replay.

## Limits: read before using these numbers

* **`submission.csv` is built on a file tuned on the public rows.** It is +46 on the board since version 4 and about -56 on
  out-of-fold AUC, measured on 700,000 rows (version 5 was at -37; step 9 gave 13 back; steps 10 to 13 together about -31). The public rows are 60,000 of the
  300,000 test rows; the private score is computed on the other 240,000. For the private rows we would take `submission_cv.csv`.
* **The public model is not shipped.** It is fitted on scored submissions: 44 of ours in version 4, 69 files by the last step of
  version 5, 76 by step 9, 95 by step 11, 99 by step 12 and 105 after step 13 (ours, and public files whose authors state a score). This notebook recomputes the files, the replay and the CV changes
  from public inputs; the forecasts and their errors are reported as we measured them.
* **The CV side is a replay, not the file's CV.** About 1% of the 0.95370 file has no out-of-fold counterpart here.
  The moves themselves are exact: every member has real out-of-fold predictions.
* **Part of the public gain is fitted to the public rows.** That is what a public model does. A move the size of the bold mix shifts
  the AUC of a 60,000-row sample by about ±13 points by chance, so +16 alone would prove little. The claim rests on the CV change,
  measured on 700,000 rows with weights that never saw the scoring fold.
* **What CV alone supports stops at about 0.95388.** The fourth probe of version 4 and the first two of version 5 show it. Above
  that, steps 6, 8 and 9 are the ones out-of-fold AUC supports as well; the other steps were bought from the public rows.
* **This is not a verdict on anyone's file.** Public probing is how every ladder here was built, ours included.

## Every probe of version 10 (11-17)

| file | public LB |
|---|---|
| **step 13: member step (r20) + 1.0 r (this version)** | **0.95432** |
| step 13, larger: 1.5 x the member step + 1.0 r + 0.25 (A - O) | 0.95432 |
| step 13 + a CV-led step (nested +8.2, all five folds; forecast -0.3) | 0.95431 |

## Every probe of version 9 (11-17)

| file | public LB |
|---|---|
| step 12: member step (r19) + 0.3 (B - step 11) + 0.5 (A - O) (`submission_step12.csv`) | 0.95430 |

## Every probe of version 8 (11-16)

| file | public LB |
|---|---|
| step 10, balanced: CV-led weight on the stack step | 0.95424 |
| step 10 (`submission_step10.csv`) | 0.95426 |
| step 11 (`submission_step11.csv`) | 0.95427 |
| step 11 with 20% of BhuvanNR's 0.95426 file: not used (an unpinned input, 6 more points of OOF given up) | 0.95428 |

## Every probe of version 7 (11-15)

| file | public LB |
|---|---|
| **the CV-backed file: 15% of megayak's LM moved to the two GPT-2 seeds (`submission_cv.csv`)** | **0.95379** |
| the same with a further 5% towards the best single GLM: +1.4 more on OOF, inside the noise, not kept | 0.95376 |

## Every probe of version 6 (11-16)

| file | public LB |
|---|---|
| step 8: CV and forecast, from the 0.95417 file | 0.95418 |
| **step 9: the same question with one more equation (this version)** | **0.95421** |

## Every probe of version 5 (11-14 and 11-15)

| file | public LB |
|---|---|
| the bold mix (version 4) | 0.95386 |
| `bold + 0.5 (bold - balanced)`: Areeb Ali's file, rebuilt here as the starting point (his score, not submitted by us) | 0.95388 |
| joint step | 0.95388 |
| CV-led step (`submission_cv.csv`) | 0.95387 |
| step 1 | 0.95390 |
| step 2, half size | 0.95393 |
| step 2 | 0.95396 |
| step 3 | 0.95404 |
| step 4 | 0.95408 |
| step 5 (11-15): forecast +1.4, not kept | 0.95407 |
| step 6 (11-15): CV in the lead | 0.95412 |
| step 7 (11-15, version 5) | 0.95417 |

## Every probe of version 4 (11-13)

| file | public LB |
|---|---|
| the 0.95370 file (Aman Atar), and the same with our LM swap (v3) | 0.95370 |
| 90% file + 10% our wide LM stack | 0.95368 |
| 75% file + 25% our wide LM stack | 0.95362 |
| careful mix | 0.95376 |
| balanced mix | 0.95380 |
| bold mix | 0.95386 |
| bold mix + one more step | 0.95385 |

Earlier probes are in the version history of this notebook.

## Credits

**Aman Atar** (the 0.95370 file, the bundle of 16 streams, Strike 10), and the authors of the streams in it:
**megayak**, **masayakawamata**, **slimypunk03**, **sadamtorres**, **yekenot**, **alexchilton**, **daoviet**, **sameershaikh27**.
**Asterios Panoras** ("How Much of the Public Top Is Real?": measuring the noise of a public gap on out-of-fold rows),
**Areeb Ali** (Bold, Then Bolder: the 0.95388 step and eight scored probes of our mixes; and, with **samanyu1808**, the LM steps on the ladder), **Moran Z** (Strike 9), **Matapanino** (GPT-2 target-first code),
**Chris Deotte** (the target-first recipe, S6E10).
Version 6: **BhuvanNR** (an open rebuild of the public model and one more step, 0.95419), **Areeb Ali** (Two Families, One Blend,
0.95421), **Asterios Panoras** (This Score Is Not Real, 0.95422, with the out-of-fold cost of every step) and **alexchilton** (the
shallow-tree stack in those files, and a replay of this ladder on labelled rows): their scored files are three of the equations
behind step 9.
Version 8: **alexchilton** (the shallow-tree stack, member 88 of step 10), **Asterios Panoras** (the steered 0.95422 file, the move
`A - O`, the 0.95424 file, and the pinned copies in
[s6e11-public-board-files](https://www.kaggle.com/datasets/asteriospanoras/s6e11-public-board-files) that make this version exact),
**Areeb Ali** (Two Families, One Blend and Steer, Then Mix: fourteen scored probes), **BhuvanNR** (Fable & Two Families: 0.95420 and
0.95426) and **Malyshev Danil** (the 80/20 mix). Their stated scores are sixteen of the equations behind steps 10 and 11.
Version 9: **Areeb Ali** (Two Ladders, One Average, 0.95428) and **Asterios Panoras** (0.95428, and the pinned copy of BhuvanNR's
step-10 file in his dataset): equations 98 and 99, and the two directions inside step 12.
Version 10: **Areeb Ali** (Split, Then Push, 0.95431: the split of our step 12 and the direction `r`, and the 0.95429 probe) and
**Malyshev Danil** (the same file scored again from his notebook): equations 100 and 101.
If this was useful, please upvote them too.